# Movie Clustering Walkthrough

**Goal:** Group ~8,000 movies into clusters based on 248 content features (themes, settings, moods, character types, etc.) so that similar movies end up in the same group.

### What is clustering?
Clustering is an **unsupervised** machine learning technique — meaning we don't give the algorithm any labels (like "action" or "comedy"). Instead, it finds natural groupings in the data on its own by looking at how similar movies are to each other across all their features.

### Our roadmap
1. **Load & explore** the data
2. **Build a feature matrix** (pivot long-form → one row per movie)
3. **Preprocess** (scale features so no single one dominates)
4. **Choose K** — how many clusters? (Elbow method + Silhouette scores)
5. **Run K-Means** clustering
6. **Analyze** what defines each cluster
7. **Visualize** with PCA and t-SNE

Let's get started!

---
## Step 1 — Import Libraries

We need `pandas` for data handling, `scikit-learn` for clustering, and `matplotlib`/`seaborn` for visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')

# Make plots look nice
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print('All libraries loaded!')

---
## Step 2 — Load the Data

We have three CSV files:
- **feature_data_longform.csv** — Each row says: *movie X has feature Y with intensity Z*
- **feature_taxonomy.csv** — Maps feature IDs to human-readable names
- **genre_data.csv** — Movie titles and genre labels

In [ ]:
# Load all three files
features_long = pd.read_csv('feature_data_longform.csv')
taxonomy      = pd.read_csv('feature_taxonomy.csv')
genres        = pd.read_csv('genre_data.csv')

print(f'Feature data:  {features_long.shape[0]:,} rows  (movie-feature pairs)')
print(f'Taxonomy:      {taxonomy.shape[0]} features')
print(f'Genre data:    {genres.shape[0]:,} movies')

### 2a — Peek at each dataset

In [ ]:
print('=== Feature Data (long form) ===')
display(features_long.head(10))
print(f'\nUnique movies:   {features_long["imdb_id"].nunique():,}')
print(f'Unique features: {features_long["feature_id"].nunique()}')
print(f'Trigger range:   {features_long["trigger"].min()} to {features_long["trigger"].max()}')

In [ ]:
print('=== Feature Taxonomy ===')
display(taxonomy.head(15))
print(f'\nSample features: {", ".join(taxonomy["feature"].sample(8).tolist())}')

In [ ]:
print('=== Genre Data ===')
display(genres.head(10))

---
## Step 3 — Build the Feature Matrix

Right now our feature data is in **long form** — one row per movie-feature pair. For clustering we need **wide form**: one row per movie, one column per feature.

We'll also swap the numeric `feature_id` for the human-readable feature name so our results are easier to interpret.

In [ ]:
# Merge feature names into the long-form data
features_long = features_long.merge(taxonomy, on='feature_id', how='left')

# Pivot: rows = movies, columns = feature names, values = trigger intensity
feature_matrix = features_long.pivot_table(
    index='imdb_id',
    columns='feature',
    values='trigger',
    aggfunc='first'       # one value per movie-feature pair
).fillna(0)               # if a movie is missing a feature, treat it as 0

print(f'Feature matrix shape: {feature_matrix.shape}')
print(f'  → {feature_matrix.shape[0]:,} movies  x  {feature_matrix.shape[1]} features')
display(feature_matrix.iloc[:5, :8])  # preview a small corner

### 3b — Add Genre Features (One-Hot Encoded)

Each movie has one or more genre labels like "Action, Comedy, Drama". We can turn these into **binary columns** — one per genre — where 1 means the movie belongs to that genre and 0 means it doesn't. This is called **one-hot encoding**.

**Will these scale correctly with the other features?** Yes. The trigger features range from 0–3 and genres are 0/1, but `StandardScaler` handles this just fine — it transforms *every* column to mean=0, std=1 regardless of the original range. After scaling, a genre column carries comparable weight to any trigger column.

In [ ]:
# One-hot encode genres and merge into the feature matrix
# Step 1: Split multi-genre strings into individual genres per movie
# reset_index() gives each row a unique index after explode (needed for newer pandas)
genre_expanded = (
    genres.assign(genre_list=genres['genre'].str.split(', '))
    .explode('genre_list')
    .reset_index(drop=True)
)

# Step 2: Create a binary pivot table (1 = movie has this genre, 0 = doesn't)
genre_onehot = pd.crosstab(genre_expanded['movie_id'], genre_expanded['genre_list'])

# Add a prefix so genre columns are easy to identify later
genre_onehot.columns = ['genre_' + col for col in genre_onehot.columns]
genre_onehot.index.name = 'imdb_id'

print(f'Genre one-hot shape: {genre_onehot.shape}')
print(f'  → {genre_onehot.shape[1]} unique genre columns\n')
display(genre_onehot.head())

# Step 3: Merge with our feature matrix (join on the shared imdb_id index)
# Drop any existing genre_ columns first (safe to re-run this cell)
existing_genre_cols = [c for c in feature_matrix.columns if c.startswith('genre_')]
if existing_genre_cols:
    feature_matrix = feature_matrix.drop(columns=existing_genre_cols)

feature_matrix = feature_matrix.join(genre_onehot, how='left').fillna(0)

print(f'\nUpdated feature matrix: {feature_matrix.shape}')
print(f'  → {feature_matrix.shape[0]:,} movies  x  {feature_matrix.shape[1]} features')
print(f'  → {len([c for c in feature_matrix.columns if c.startswith("genre_")])} genre columns + {len([c for c in feature_matrix.columns if not c.startswith("genre_")])} trigger columns')

---
## Step 4 — Preprocessing: Scale the Features

**Why scale?** Different features might live on different ranges. If one feature goes from 0–100 and another from 0–3, K-Means will think differences in the first feature are much more "important" just because they're bigger numbers.

`StandardScaler` transforms each feature so it has **mean = 0** and **std = 1**. This puts every feature on an equal footing.

In [ ]:
# Check the distribution of trigger values before scaling
print('Before scaling — trigger value statistics:')
print(feature_matrix.stack().describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribution of all trigger values
feature_matrix.stack().hist(bins=30, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of ALL trigger values')
axes[0].set_xlabel('Trigger intensity')

# How many non-zero features does each movie have?
(feature_matrix > 0).sum(axis=1).hist(bins=40, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Non-zero features per movie')
axes[1].set_xlabel('Number of active features')

plt.tight_layout()
plt.show()

In [ ]:
# Range of values for each feature (min, max, and spread)
feature_ranges = pd.DataFrame({
    'min': feature_matrix.min(),
    'max': feature_matrix.max(),
    'range': feature_matrix.max() - feature_matrix.min(),
    'mean': feature_matrix.mean(),
    'std': feature_matrix.std()
}).sort_values('range', ascending=False)

print(f'Value ranges across all {len(feature_ranges)} features:\n')
print(f'{"Feature":<50s} {"Min":>6s} {"Max":>6s} {"Range":>7s} {"Mean":>7s} {"Std":>7s}')
print('-' * 85)
for feat, row in feature_ranges.iterrows():
    print(f'{feat:<50s} {row["min"]:>6.1f} {row["max"]:>6.1f} {row["range"]:>7.1f} {row["mean"]:>7.2f} {row["std"]:>7.2f}')

# Quick summary
print(f'\n{"="*85}')
print(f'Unique ranges found: {sorted(feature_ranges["range"].unique())}')
print(f'Features with range 0 (constant): {(feature_ranges["range"] == 0).sum()}')

In [ ]:
# Keep both raw and standardized versions available for later toggles
X_raw = feature_matrix.to_numpy(dtype=np.float64, copy=True)

scaler = StandardScaler()
X_standardized = scaler.fit_transform(X_raw)

# Active matrix used by downstream cells (default: standardized)
X_scaled = X_standardized.copy()

print(f'Raw matrix shape: {X_raw.shape}')
print(f'Standardized matrix shape: {X_standardized.shape}')
print(f'Mean of first standardized feature (should be ~0): {X_standardized[:, 0].mean():.4f}')
print(f'Std of first standardized feature  (should be ~1): {X_standardized[:, 0].std():.4f}')
print('\nPrepared both raw and standardized feature matrices.')

### Step 4b — Switch to Cosine Distance (L2 Normalization)

K-Means uses **Euclidean distance** by default — the straight-line distance between two points. But for movie feature profiles, **cosine distance** is usually a better fit. Here's why:

- **Euclidean** cares about *magnitude*: a movie with all features at 2.0 and one with all features at 1.0 are "far apart" even though they have the exact same pattern.
- **Cosine** cares about *direction*: it measures the angle between two feature vectors. Two movies with the same relative pattern of features are "close" regardless of overall intensity.

This matters for your data because some movies may have generally higher trigger scores across the board (e.g., blockbusters with strong everything) vs. quieter films — but the *shape* of their feature profile is what really defines their type.

**The trick:** If we **L2-normalize** each row (scale each movie's vector to unit length), then Euclidean distance becomes mathematically equivalent to cosine distance. This lets us keep using standard K-Means while getting cosine behavior.

Set `USE_COSINE = True` below to enable this. Set it to `False` to stick with plain Euclidean.

In [ ]:
from sklearn.preprocessing import normalize

# ============================================
# TOGGLES: FEATURE SCALE + COSINE BEHAVIOR
# ============================================
USE_STANDARD_SCALING = True   # True = z-scored features, False = raw features
USE_COSINE = True             # True = L2-normalize rows, False = no L2 normalization

# Rebuild active matrix from source each run so toggles are reversible
X_scaled = X_standardized.copy() if USE_STANDARD_SCALING else X_raw.copy()
print('Base matrix:', 'standardized (z-score)' if USE_STANDARD_SCALING else 'raw (unscaled)')

if USE_COSINE:
    # After L2 normalization, Euclidean distance behaves like cosine distance
    X_scaled = normalize(X_scaled, norm='l2')
    print('L2 normalization applied — K-Means will now use cosine-like distance.')
    print(f'  Row norms (should all be 1.0): {np.linalg.norm(X_scaled[:3], axis=1).round(4)}')
else:
    print('No L2 normalization — using plain Euclidean geometry on the base matrix.')

print(f'\nFinal input matrix shape: {X_scaled.shape}')

---
## Step 5 — Compute PCA for Visualization

Our data lives in 269 dimensions. Throughout this notebook we'll use **PCA** (Principal Component Analysis) to project movies down to 2D for plotting. PCA finds the two directions that capture the most variance — think of it as the best possible "shadow" of a high-dimensional object.

We compute this once here so every method below can reuse the same 2D projection for fair visual comparison.

In [ ]:
# Compute PCA once — reused by every visualization below
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f'Variance explained by 2 components: {pca.explained_variance_ratio_.sum():.1%}')
print(f'  PC1: {pca.explained_variance_ratio_[0]:.1%}')
print(f'  PC2: {pca.explained_variance_ratio_[1]:.1%}')
print(f'\nNote: This captures only ~{pca.explained_variance_ratio_.sum():.0%} of the total variance.')
print('Clustering still uses all 269 dimensions — PCA is just for visualization.')

---
## Step 5b — Dimensionality Reduction for Cluster Separation

### The Problem: Curse of Dimensionality

Our data has **269 features**. While PCA helps us *visualize* clusters in 2D, all clustering algorithms still work in the full 269-dimensional space. This creates several problems:

1. **Curse of Dimensionality**: In high dimensions, distances become less meaningful — everything is far from everything else
2. **Noise accumulation**: Many features may just add noise rather than signal
3. **Computational cost**: Clustering 269 dimensions is slower

### The Solution: Dimensionality Reduction for Clustering

Instead of clustering in 269D, we can:
- Reduce to 20-50 dimensions using techniques like PCA or UMAP
- Remove noise while keeping signal
- Improve cluster separation and quality

**Key difference**: 
- **Visualization**: 2D PCA to plot clusters (what we did in Step 5)
- **Cluster Separation**: 20-50D reduction to improve clustering quality (what we do here)

We'll test:
1. **PCA** with varying components (10, 20, 30, 50)
2. **UMAP** (preserves local structure better than PCA)
3. Measure cluster quality with silhouette score, Davies-Bouldin index, etc.

In [ ]:
# Dimensionality reduction imports
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import umap  # Install with: pip install umap-learn
from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# First, let's see how much variance we can capture with different numbers of components
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

# Calculate cumulative variance explained
cumsum_variance = np.cumsum(pca_full.explained_variance_ratio_)

# Find how many components needed for 80%, 90%, 95% variance
for threshold in [0.80, 0.90, 0.95]:
    n_comp = np.argmax(cumsum_variance >= threshold) + 1
    print(f'{threshold:.0%} variance explained by {n_comp} components')

# Plot cumulative variance
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(cumsum_variance) + 1), cumsum_variance, linewidth=2)
ax.axhline(0.80, color='red', linestyle='--', alpha=0.7, label='80% variance')
ax.axhline(0.90, color='orange', linestyle='--', alpha=0.7, label='90% variance')
ax.axhline(0.95, color='green', linestyle='--', alpha=0.7, label='95% variance')
ax.set_xlabel('Number of Components', fontsize=12)
ax.set_ylabel('Cumulative Variance Explained', fontsize=12)
ax.set_title('PCA Variance Explained: How Many Dimensions Do We Need?', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 100)  # Focus on first 100 components
plt.tight_layout()
plt.show()

print(f'\nOriginal dimensions: {X_scaled.shape[1]}')

### Dimensionality Reduction Techniques to Test

**PCA (Principal Component Analysis)**
- Linear technique that finds directions of maximum variance
- Fast and deterministic
- Best for: Data with linear relationships
- We'll test: 10, 20, 30, 50 components

**UMAP (Uniform Manifold Approximation and Projection)**
- Non-linear technique that preserves local and global structure
- Often better than PCA for clustering
- Best for: Complex, non-linear relationships
- We'll test: 10, 20, 30, 50 components

**Cluster Quality Metrics**:
- **Silhouette Score** (higher is better, -1 to 1): Measures how similar points are to their own cluster vs others
- **Davies-Bouldin Index** (lower is better): Average similarity between clusters
- **Calinski-Harabasz Score** (higher is better): Ratio of between-cluster to within-cluster variance

In [ ]:
# =============================================
# APPLY MULTIPLE DIMENSIONALITY REDUCTION TECHNIQUES
# =============================================

n_components_to_test = [10, 20, 30, 50]
reduced_data = {}

print('Applying dimensionality reduction techniques...\n')

# Baseline: Original data
reduced_data['Original (269D)'] = X_scaled
print(f'✓ Original data: {X_scaled.shape}')

# PCA with different n_components
for n in n_components_to_test:
    pca_reducer = PCA(n_components=n, random_state=42)
    X_pca_reduced = pca_reducer.fit_transform(X_scaled)
    variance_retained = pca_reducer.explained_variance_ratio_.sum()
    reduced_data[f'PCA ({n}D)'] = X_pca_reduced
    print(f'✓ PCA {n}D: {X_pca_reduced.shape} — retains {variance_retained:.1%} variance')

# UMAP with different n_components
for n in n_components_to_test:
    umap_reducer = umap.UMAP(n_components=n, random_state=42, n_neighbors=15, min_dist=0.1)
    X_umap_reduced = umap_reducer.fit_transform(X_scaled)
    reduced_data[f'UMAP ({n}D)'] = X_umap_reduced
    print(f'✓ UMAP {n}D: {X_umap_reduced.shape}')

print(f'\nTotal methods: {len(reduced_data)}')

In [ ]:
# =============================================
# EVALUATE CLUSTER QUALITY FOR EACH REDUCED SPACE
# =============================================

# Test with K-Means (K=8, matching your analysis)
K_TEST = 8  # Adjust to match your chosen K

results = []

print(f'Testing K-Means clustering (K={K_TEST}) on each reduced space...\n')

for method_name, X_reduced in reduced_data.items():
    # Run K-Means
    kmeans = KMeans(n_clusters=K_TEST, random_state=42, n_init=20)
    labels = kmeans.fit_predict(X_reduced)
    
    # Calculate quality metrics
    sil_score = silhouette_score(X_reduced, labels)
    db_score = davies_bouldin_score(X_reduced, labels)
    ch_score = calinski_harabasz_score(X_reduced, labels)
    
    results.append({
        'Method': method_name,
        'Silhouette': sil_score,
        'Davies-Bouldin': db_score,
        'Calinski-Harabasz': ch_score
    })
    
    print(f'{method_name:20s} | Silhouette: {sil_score:6.3f} | DB: {db_score:6.3f} | CH: {ch_score:9.1f}')

results_df = pd.DataFrame(results)

In [ ]:
# =============================================
# VISUALIZE CLUSTER QUALITY COMPARISON
# =============================================

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Separate by technique type
results_df['Technique'] = results_df['Method'].apply(
    lambda x: 'Original' if 'Original' in x else ('PCA' if 'PCA' in x else 'UMAP')
)
results_df['Dimensions'] = results_df['Method'].apply(
    lambda x: 269 if 'Original' in x else int(x.split('(')[1].split('D')[0])
)

# Plot 1: Silhouette Score (higher is better)
for technique in ['Original', 'PCA', 'UMAP']:
    subset = results_df[results_df['Technique'] == technique]
    axes[0].plot(subset['Dimensions'], subset['Silhouette'], 'o-', label=technique, markersize=8, linewidth=2)
axes[0].set_xlabel('Number of Dimensions', fontsize=12)
axes[0].set_ylabel('Silhouette Score', fontsize=12)
axes[0].set_title('Silhouette Score (Higher is Better)', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Davies-Bouldin Index (lower is better)
for technique in ['Original', 'PCA', 'UMAP']:
    subset = results_df[results_df['Technique'] == technique]
    axes[1].plot(subset['Dimensions'], subset['Davies-Bouldin'], 'o-', label=technique, markersize=8, linewidth=2)
axes[1].set_xlabel('Number of Dimensions', fontsize=12)
axes[1].set_ylabel('Davies-Bouldin Index', fontsize=12)
axes[1].set_title('Davies-Bouldin Index (Lower is Better)', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Calinski-Harabasz Score (higher is better)
for technique in ['Original', 'PCA', 'UMAP']:
    subset = results_df[results_df['Technique'] == technique]
    axes[2].plot(subset['Dimensions'], subset['Calinski-Harabasz'], 'o-', label=technique, markersize=8, linewidth=2)
axes[2].set_xlabel('Number of Dimensions', fontsize=12)
axes[2].set_ylabel('Calinski-Harabasz Score', fontsize=12)
axes[2].set_title('Calinski-Harabasz Score (Higher is Better)', fontsize=14, fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================
# IDENTIFY BEST METHODS
# =============================================

print('=' * 80)
print('BEST METHODS BY METRIC')
print('=' * 80)

# Best Silhouette (highest)
best_sil = results_df.loc[results_df['Silhouette'].idxmax()]
print(f"\n🏆 Best Silhouette Score: {best_sil['Method']}")
print(f"   Score: {best_sil['Silhouette']:.4f}")

# Best Davies-Bouldin (lowest)
best_db = results_df.loc[results_df['Davies-Bouldin'].idxmin()]
print(f"\n🏆 Best Davies-Bouldin Index: {best_db['Method']}")
print(f"   Score: {best_db['Davies-Bouldin']:.4f}")

# Best Calinski-Harabasz (highest)
best_ch = results_df.loc[results_df['Calinski-Harabasz'].idxmax()]
print(f"\n🏆 Best Calinski-Harabasz Score: {best_ch['Method']}")
print(f"   Score: {best_ch['Calinski-Harabasz']:.1f}")

# Overall recommendation (majority vote)
print('\n' + '=' * 80)
print('RECOMMENDATION')
print('=' * 80)
print('\nBased on the metrics above, consider using one of the top-performing methods')
print('for your clustering instead of the original 269D space.')
print('\nTypically:')
print('  • UMAP preserves cluster structure better than PCA')
print('  • 20-30 dimensions often gives best balance of noise reduction and signal retention')
print('  • More dimensions (50+) approaches original performance but with less noise')

# Display full comparison table
print('\n' + '=' * 80)
print('FULL COMPARISON TABLE')
print('=' * 80)
print(results_df.to_string(index=False))

### Step 5b-ii — UMAP Embedding Quality: How Well Is Structure Preserved?

Cluster quality metrics (Silhouette, DB, CH) tell us how well clusters separate **within** the reduced space — but they don't tell us whether the reduction itself **faithfully preserves** the original high-dimensional relationships.

Three complementary metrics address this:

1. **Pearson Correlation of Pairwise Distances** — Do movies that were far apart (or close together) in 269D stay that way in the reduced space? A correlation near 1.0 means the embedding preserves global distance structure.

2. **Trustworthiness** — Are a point's nearest neighbors in the reduced space *actually* close in the original space? High trustworthiness (→ 1.0) means the embedding doesn't introduce false neighbors.

3. **Neighborhood Preservation** — Of a point's true nearest neighbors in 269D, how many remain neighbors after reduction? High preservation means the embedding doesn't lose real neighbors.

In [ ]:
from sklearn.metrics import pairwise_distances, trustworthiness
from sklearn.neighbors import NearestNeighbors
from scipy.stats import pearsonr

# =============================================
# UMAP EMBEDDING QUALITY EVALUATION
# =============================================
# Compare each reduced representation against the original 269D space
# using structure-preservation metrics (not just cluster quality).

# Sample for speed — pairwise distances on 8000 points is expensive
SAMPLE_SIZE = 2000
np.random.seed(42)
sample_idx = np.random.choice(X_scaled.shape[0], SAMPLE_SIZE, replace=False)
X_orig_sample = X_scaled[sample_idx]

# Pairwise distances in original space (computed once)
dist_orig = pairwise_distances(X_orig_sample).ravel()

K_NEIGHBORS = 15  # neighborhood size for preservation metrics

# Compute metrics for each reduced dataset
quality_results = []
methods_to_evaluate = {k: v for k, v in reduced_data.items() if k \!= 'Original (269D)'}

print(f'Evaluating embedding quality ({SAMPLE_SIZE} sampled movies, K={K_NEIGHBORS} neighbors)...
')

for method_name, X_reduced in methods_to_evaluate.items():
    X_red_sample = X_reduced[sample_idx]

    # 1. Pearson correlation of pairwise distances
    dist_red = pairwise_distances(X_red_sample).ravel()
    corr, _ = pearsonr(dist_orig, dist_red)

    # 2. Trustworthiness (sklearn built-in)
    trust = trustworthiness(X_orig_sample, X_red_sample, n_neighbors=K_NEIGHBORS)

    # 3. Neighborhood Preservation
    nn_orig = NearestNeighbors(n_neighbors=K_NEIGHBORS).fit(X_orig_sample)
    nn_red = NearestNeighbors(n_neighbors=K_NEIGHBORS).fit(X_red_sample)
    orig_neighbors = set(map(tuple, nn_orig.kneighbors(X_orig_sample, return_distance=False).tolist()))
    red_neighbors = set(map(tuple, nn_red.kneighbors(X_red_sample, return_distance=False).tolist()))

    # Per-point preservation: fraction of original neighbors retained
    orig_nb = nn_orig.kneighbors(X_orig_sample, return_distance=False)
    red_nb = nn_red.kneighbors(X_red_sample, return_distance=False)
    preservation_scores = []
    for i in range(len(X_orig_sample)):
        overlap = len(set(orig_nb[i]) & set(red_nb[i]))
        preservation_scores.append(overlap / K_NEIGHBORS)
    neighborhood_pres = np.mean(preservation_scores)

    quality_results.append({
        'Method': method_name,
        'Pearson Corr': corr,
        'Trustworthiness': trust,
        'Neighborhood Pres': neighborhood_pres
    })
    print(f'{method_name:20s} | Pearson: {corr:.4f} | Trust: {trust:.4f} | Nbr Pres: {neighborhood_pres:.4f}')

quality_df = pd.DataFrame(quality_results)

# Add technique and dimension columns for plotting
quality_df['Technique'] = quality_df['Method'].apply(
    lambda x: 'PCA' if 'PCA' in x else ('UMAP' if 'UMAP' in x else 'Other')
)
quality_df['Dimensions'] = quality_df['Method'].apply(
    lambda x: int(x.split('(')[1].split('D')[0])
)

print(f'
Done\!')

In [ ]:
# =============================================
# VISUALIZE EMBEDDING QUALITY METRICS
# =============================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# --- Top-left: Pearson Correlation by method ---
for technique in ["PCA", "UMAP"]:
    subset = quality_df[quality_df["Technique"] == technique]
    axes[0, 0].plot(subset["Dimensions"], subset["Pearson Corr"], "o-", label=technique, markersize=8, linewidth=2)
axes[0, 0].set_xlabel("Number of Dimensions", fontsize=12)
axes[0, 0].set_ylabel("Pearson Correlation", fontsize=12)
axes[0, 0].set_title("Pairwise Distance Correlation with Original Space", fontsize=13, fontweight="bold")
axes[0, 0].set_ylim(0, 1.05)
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(True, alpha=0.3)

# --- Top-right: Trustworthiness by method ---
for technique in ["PCA", "UMAP"]:
    subset = quality_df[quality_df["Technique"] == technique]
    axes[0, 1].plot(subset["Dimensions"], subset["Trustworthiness"], "s-", label=technique, markersize=8, linewidth=2)
axes[0, 1].set_xlabel("Number of Dimensions", fontsize=12)
axes[0, 1].set_ylabel("Trustworthiness", fontsize=12)
axes[0, 1].set_title("Trustworthiness (Are Reduced Neighbors Real?)", fontsize=13, fontweight="bold")
axes[0, 1].set_ylim(0, 1.05)
axes[0, 1].legend(fontsize=11)
axes[0, 1].grid(True, alpha=0.3)

# --- Bottom-left: Neighborhood Preservation by method ---
for technique in ["PCA", "UMAP"]:
    subset = quality_df[quality_df["Technique"] == technique]
    axes[1, 0].plot(subset["Dimensions"], subset["Neighborhood Pres"], "D-", label=technique, markersize=8, linewidth=2)
axes[1, 0].set_xlabel("Number of Dimensions", fontsize=12)
axes[1, 0].set_ylabel("Neighborhood Preservation", fontsize=12)
axes[1, 0].set_title("Neighborhood Preservation (Are Original Neighbors Kept?)", fontsize=13, fontweight="bold")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].legend(fontsize=11)
axes[1, 0].grid(True, alpha=0.3)

# --- Bottom-right: Scatter plot of pairwise distances (best UMAP vs original) ---
# Pick the UMAP method with highest trustworthiness for the scatter
umap_rows = quality_df[quality_df["Technique"] == "UMAP"]
if len(umap_rows) > 0:
    best_umap = umap_rows.loc[umap_rows["Trustworthiness"].idxmax(), "Method"]
    X_best_sample = reduced_data[best_umap][sample_idx]
    dist_best = pairwise_distances(X_best_sample).ravel()
    # Subsample pairs for plotting (full pairwise is millions of points)
    n_pairs = len(dist_orig)
    plot_idx = np.random.choice(n_pairs, min(50000, n_pairs), replace=False)
    axes[1, 1].scatter(dist_orig[plot_idx], dist_best[plot_idx], alpha=0.05, s=1, color="steelblue")
    corr_val = quality_df.loc[quality_df["Method"] == best_umap, "Pearson Corr"].values[0]
    axes[1, 1].set_xlabel("Pairwise Distance (Original 269D)", fontsize=12)
    axes[1, 1].set_ylabel(f"Pairwise Distance ({best_umap})", fontsize=12)
    axes[1, 1].set_title(f"Distance Correlation: {best_umap} (r={corr_val:.4f})", fontsize=13, fontweight="bold")
    axes[1, 1].grid(True, alpha=0.3)
    # Add diagonal reference line
    lims = [min(dist_orig[plot_idx].min(), dist_best[plot_idx].min()),
            max(dist_orig[plot_idx].max(), dist_best[plot_idx].max())]
    axes[1, 1].plot(lims, lims, "r--", alpha=0.5, label="Perfect correlation")
    axes[1, 1].legend(fontsize=10)

plt.suptitle("UMAP vs PCA: Embedding Quality Metrics", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# Summary table
print("
" + "=" * 85)
print("EMBEDDING QUALITY SUMMARY")
print("=" * 85)
print(f"{"Method":<20s} {"Pearson Corr":>14s} {"Trustworthiness":>17s} {"Nbr Preservation":>18s}")
print("-" * 85)
for _, row in quality_df.iterrows():
    print(f"{row["Method"]:<20s} {row["Pearson Corr"]:>14.4f} {row["Trustworthiness"]:>17.4f} {row["Neighborhood Pres"]:>18.4f}")
print("
Interpretation:")
print("  Pearson Corr  -> 1.0: global distance structure is well preserved")
print("  Trustworthy   -> 1.0: neighbors in reduced space are real neighbors")
print("  Nbr Pres      -> 1.0: original neighbors are still neighbors after reduction")


---
### How to Use Dimensionality Reduction in Your Clustering

Based on the results above, you can now **replace `X_scaled`** with one of the reduced datasets when running clustering algorithms.

**Example**: If UMAP (30D) performed best, add this before your clustering code:

```python
# Use UMAP-reduced space for clustering
X_clustering = reduced_data['UMAP (30D)']

# Then run K-Means (or any clustering) on X_clustering
kmeans = KMeans(n_clusters=8, random_state=42)
labels = kmeans.fit_predict(X_clustering)
```

**Toggle**: Add this cell right before Step 6 to easily switch between original and reduced spaces:

```python
# =============================================
# DIMENSIONALITY REDUCTION TOGGLE
# =============================================
USE_DIMENSIONALITY_REDUCTION = True
DR_METHOD = 'UMAP (30D)'  # Choose from results_df['Method']

if USE_DIMENSIONALITY_REDUCTION:
    X_clustering = reduced_data[DR_METHOD]
    print(f'Using {DR_METHOD} for clustering: {X_clustering.shape}')
else:
    X_clustering = X_scaled
    print(f'Using original space for clustering: {X_clustering.shape}')
```

Then replace `X_scaled` with `X_clustering` in all your clustering code below!

---
## Step 5c — Separation Expansion Techniques

### What is Separation Expansion?

Beyond dimensionality reduction, we can also **expand** the separation between clusters using:

1. **Distance Metric Tuning**: Different distance metrics emphasize different aspects
2. **Feature Weighting**: Give more importance to discriminative features
3. **Kernel Methods**: Map data to higher-dimensional spaces where clusters separate better
4. **Spectral Embedding**: Use graph-based methods to enhance cluster boundaries

Let's test a few techniques:

In [ ]:
from sklearn.manifold import SpectralEmbedding
from sklearn.neighbors import kneighbors_graph

# =============================================
# SPECTRAL EMBEDDING FOR SEPARATION EXPANSION
# =============================================
# Spectral embedding uses graph connectivity to find embeddings
# that maximize cluster separation

print('Applying Spectral Embedding...')

# Test different dimensions
spectral_results = []

for n in [10, 20, 30]:
    print(f'  Testing {n} dimensions...', end=' ')
    
    # Create spectral embedding
    spectral = SpectralEmbedding(n_components=n, random_state=42, n_neighbors=10)
    X_spectral = spectral.fit_transform(X_scaled)
    
    # Evaluate clustering quality
    kmeans = KMeans(n_clusters=K_TEST, random_state=42, n_init=20)
    labels = kmeans.fit_predict(X_spectral)
    
    sil = silhouette_score(X_spectral, labels)
    db = davies_bouldin_score(X_spectral, labels)
    ch = calinski_harabasz_score(X_spectral, labels)
    
    spectral_results.append({
        'Method': f'Spectral ({n}D)',
        'Silhouette': sil,
        'Davies-Bouldin': db,
        'Calinski-Harabasz': ch
    })
    
    # Store for later use
    reduced_data[f'Spectral ({n}D)'] = X_spectral
    
    print(f'Sil: {sil:.3f}, DB: {db:.3f}, CH: {ch:.1f}')

spectral_df = pd.DataFrame(spectral_results)
print('\n✓ Spectral embedding complete')

In [ ]:
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.decomposition import KernelPCA

# =============================================
# KERNEL PCA FOR NON-LINEAR SEPARATION
# =============================================
# Kernel PCA applies non-linear transformations that can
# expand cluster separation in complex datasets

print('Applying Kernel PCA with RBF kernel...')

kernel_results = []

for n in [20, 30, 50]:
    print(f'  Testing {n} dimensions...', end=' ')
    
    # Apply Kernel PCA with RBF kernel
    kpca = KernelPCA(n_components=n, kernel='rbf', gamma=1/X_scaled.shape[1], random_state=42)
    X_kpca = kpca.fit_transform(X_scaled)
    
    # Evaluate clustering quality
    kmeans = KMeans(n_clusters=K_TEST, random_state=42, n_init=20)
    labels = kmeans.fit_predict(X_kpca)
    
    sil = silhouette_score(X_kpca, labels)
    db = davies_bouldin_score(X_kpca, labels)
    ch = calinski_harabasz_score(X_kpca, labels)
    
    kernel_results.append({
        'Method': f'Kernel PCA ({n}D)',
        'Silhouette': sil,
        'Davies-Bouldin': db,
        'Calinski-Harabasz': ch
    })
    
    # Store for later use
    reduced_data[f'Kernel PCA ({n}D)'] = X_kpca
    
    print(f'Sil: {sil:.3f}, DB: {db:.3f}, CH: {ch:.1f}')

kernel_df = pd.DataFrame(kernel_results)
print('\n✓ Kernel PCA complete')

In [ ]:
# =============================================
# COMBINED COMPARISON: ALL TECHNIQUES
# =============================================

# Combine all results
all_results = pd.concat([results_df, spectral_df, kernel_df], ignore_index=True)

# Sort by Silhouette score (typically most reliable metric)
all_results_sorted = all_results.sort_values('Silhouette', ascending=False)

print('=' * 80)
print('COMPLETE RANKING: ALL DIMENSIONALITY REDUCTION TECHNIQUES')
print('=' * 80)
print('\nTop 10 methods by Silhouette Score:')
print(all_results_sorted.head(10).to_string(index=False))

print('\n' + '=' * 80)
print('KEY INSIGHTS')
print('=' * 80)

best_method = all_results_sorted.iloc[0]
baseline = all_results[all_results['Method'] == 'Original (269D)'].iloc[0]

improvement_sil = (best_method['Silhouette'] - baseline['Silhouette']) / abs(baseline['Silhouette']) * 100
improvement_db = (baseline['Davies-Bouldin'] - best_method['Davies-Bouldin']) / baseline['Davies-Bouldin'] * 100

print(f"\n🏆 Best Method: {best_method['Method']}")
print(f"   Silhouette improvement: {improvement_sil:+.1f}%")
print(f"   Davies-Bouldin improvement: {improvement_db:+.1f}%")

print('\n📊 General Patterns:')
# Analyze which technique family works best
all_results['Technique'] = all_results['Method'].apply(
    lambda x: 'Original' if 'Original' in x else 
             ('PCA' if 'PCA' in x and 'Kernel' not in x else
              ('UMAP' if 'UMAP' in x else
               ('Spectral' if 'Spectral' in x else 'Kernel PCA')))
)

technique_avg = all_results.groupby('Technique')['Silhouette'].mean().sort_values(ascending=False)
print('\nAverage Silhouette Score by Technique:')
for technique, score in technique_avg.items():
    print(f'  {technique:15s}: {score:.4f}')

---
### 🎯 Final Recommendations

Based on the complete analysis:

**For Better Cluster Separation**:
1. Choose the top-performing method from the ranking above
2. Update the `DR_METHOD` variable in the toggle cell
3. Set `USE_DIMENSIONALITY_REDUCTION = True`
4. Replace `X_scaled` with `X_clustering` in your clustering code

**Typical Findings**:
- **UMAP**: Usually best for complex, non-linear relationships
- **Spectral Embedding**: Excellent when clusters have complex boundaries
- **Kernel PCA**: Good when linear PCA isn't capturing enough
- **Standard PCA**: Fast baseline, good if data is mostly linear

**Dimension Selection**:
- 10-20D: Aggressive noise reduction, may lose signal
- 20-30D: Sweet spot for most datasets
- 30-50D: Conservative, keeps most information

**Next Steps**:
1. Pick your preferred method from the rankings
2. Set it in the toggle cell before Step 6
3. Re-run all your clustering analyses (Steps 6-23)
4. Compare results to see if cluster quality and interpretability improve!

In [ ]:
# ==============================
# SET NUMBER OF FINAL CLUSTERS FOR CROSS-METHOD COMPARISON
# ==============================
# This value is used throughout the notebook to compare all methods at the same K.
# You can change it and re-run to see how results shift.
N_FINAL_CLUSTERS = 6

print(f'Cross-method comparison will use K = {N_FINAL_CLUSTERS}')

---

### 🔀 **Dimensionality Reduction Toggle**

Before running any clustering (Steps 6+), **run this cell** to choose whether to use:
- Original 269D space (default)
- Reduced space from Step 5b (recommended if you found improvement)

If using reduced space, you'll need to replace `X_scaled` with `X_clustering` in clustering code.

In [ ]:
# =============================================
# DIMENSIONALITY REDUCTION TOGGLE
# =============================================
# After running Step 5b, use this toggle to choose whether to cluster
# in the original 269D space or a reduced space

USE_DIMENSIONALITY_REDUCTION = False  # Set to True to use reduced space
DR_METHOD = 'UMAP (30D)'  # Choose from: 'PCA (10D)', 'PCA (20D)', 'PCA (30D)', 'PCA (50D)', 
                          #              'UMAP (10D)', 'UMAP (20D)', 'UMAP (30D)', 'UMAP (50D)'

if USE_DIMENSIONALITY_REDUCTION:
    # Check if dimensionality reduction has been run
    if 'reduced_data' not in globals():
        print('⚠️  ERROR: Run Step 5b first to generate reduced_data dictionary!')
    elif DR_METHOD not in reduced_data:
        print(f'⚠️  ERROR: {DR_METHOD} not found. Available methods:')
        for method in reduced_data.keys():
            print(f'    - {method}')
    else:
        X_clustering = reduced_data[DR_METHOD]
        print(f'✓ Using {DR_METHOD} for clustering')
        print(f'  Shape: {X_clustering.shape}')
        print(f'  Note: Replace X_scaled with X_clustering in your clustering code!')
else:
    X_clustering = X_scaled
    print(f'✓ Using original space (269D) for clustering')
    print(f'  Shape: {X_clustering.shape}')
    print(f'  Note: Continue using X_scaled in your clustering code as usual')

---
## Step 6 — Genre-Only Baseline

### Why start here?

Before trying any sophisticated algorithms, we need a **floor to beat**. The simplest way to group movies is by their genre labels — Action, Comedy, Drama, etc. This is how most streaming services and databases already categorize films.

If our content-feature-based clustering methods can't outperform genre-only clustering, then the 248 content features (mood, theme, setting, etc.) aren't adding any value over traditional genre labels.

### How it works

We extract just the **22 one-hot genre columns**, scale them the same way as the full matrix, and run K-Means across a range of K values. The silhouette score from this baseline becomes the **gray line** that every other method must beat in the final comparison.

In [ ]:
# Dimensionality reduction imports
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.cluster import KMeans

# UMAP is optional — requires numba, which may not support your NumPy version
try:
    import umap
    HAS_UMAP = True
    print('UMAP available.')
except ImportError:
    HAS_UMAP = False
    print('UMAP not available (numba/numpy version conflict). UMAP steps will be skipped.')

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Compute genre-only baseline metrics used by the visualization cell
from sklearn.preprocessing import StandardScaler

# Genre indicator columns (one-hot encoded)
genre_only_cols = [c for c in feature_matrix.columns if c.startswith('genre_')]
if not genre_only_cols:
    raise ValueError('No genre columns found. Expected columns like genre_action, genre_drama, etc.')

X_genre = feature_matrix[genre_only_cols].to_numpy(dtype=np.float64, copy=True)
X_genre_scaled = StandardScaler().fit_transform(X_genre)

# Sweep K to build elbow/silhouette curves
GENRE_K_RANGE = range(2, 15)
genre_inertias = []
genre_silhouettes = []

for k in GENRE_K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=20, max_iter=500)
    labels = km.fit_predict(X_genre_scaled)
    genre_inertias.append(km.inertia_)
    genre_silhouettes.append(silhouette_score(X_genre_scaled, labels, random_state=42))

best_genre_k = list(GENRE_K_RANGE)[int(np.argmax(genre_silhouettes))]

# Matched-K labels for apples-to-apples visual comparison with other methods
genre_match_labels = KMeans(
    n_clusters=N_FINAL_CLUSTERS,
    random_state=42,
    n_init=20,
    max_iter=500
).fit_predict(X_genre_scaled)

print(f'Genre baseline ready: {len(genre_only_cols)} genre columns')
print(f'Best genre-only K by silhouette: {best_genre_k}')
print(f'Matched comparison labels generated at K={N_FINAL_CLUSTERS}')

In [ ]:
# Genre-only baseline visualizations
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1. Elbow plot
axes[0].plot(list(GENRE_K_RANGE), genre_inertias, 'o-', color='gray', linewidth=1.5, markersize=4)
axes[0].set_xlabel('K', fontsize=11)
axes[0].set_ylabel('Inertia', fontsize=11)
axes[0].set_title('Genre-Only Elbow Curve', fontsize=13)

# 2. Silhouette scores
axes[1].plot(list(GENRE_K_RANGE), genre_silhouettes, 'o-', color='gray', linewidth=1.5, markersize=4)
axes[1].axvline(x=best_genre_k, color='red', linestyle='--', alpha=0.5, label=f'Best K={best_genre_k}')
axes[1].set_xlabel('K', fontsize=11)
axes[1].set_ylabel('Silhouette Score', fontsize=11)
axes[1].set_title('Genre-Only Silhouette Scores', fontsize=13)
axes[1].legend()

# 3. PCA projection (using the FULL feature PCA, not genre-only PCA)
genre_palette = sns.color_palette('Paired', N_FINAL_CLUSTERS)
for cid in range(N_FINAL_CLUSTERS):
    mask = genre_match_labels == cid
    axes[2].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=[genre_palette[cid]], alpha=0.3, s=10, edgecolors='none',
                    label=f'C{cid} ({mask.sum():,})')
axes[2].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=11)
axes[2].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=11)
axes[2].set_title(f'Genre-Only Clusters (K={N_FINAL_CLUSTERS}) — Full PCA Space', fontsize=13)
axes[2].legend(fontsize=7, markerscale=3)

plt.tight_layout()
plt.show()

# Quick genre breakdown of each genre-only cluster
print('\nGenre-only cluster profiles (top genres per cluster):')
genre_cluster_df = feature_matrix[genre_only_cols].copy()
genre_cluster_df['cluster'] = genre_match_labels

for cid in range(N_FINAL_CLUSTERS):
    cluster_data = genre_cluster_df[genre_cluster_df['cluster'] == cid]
    means = cluster_data[genre_only_cols].mean().sort_values(ascending=False)
    top3 = ', '.join([f'{g.replace("genre_", "")} ({v:.0%})' for g, v in means.head(3).items()])
    print(f'  Cluster {cid} ({len(cluster_data):,} movies): {top3}')

---
## Step 7 — K-Means Clustering

### Why K-Means first?

K-Means is the most widely used clustering algorithm and a natural first step. It's fast, intuitive, and gives us a strong foundation to compare everything else against.

### How it works

K-Means partitions movies into exactly **K groups** by:
1. Randomly placing K "centroid" points in feature space
2. Assigning every movie to its nearest centroid
3. Moving each centroid to the center of its assigned movies
4. Repeating steps 2–3 until assignments stop changing

The result: K spherical-ish clusters, each defined by the centroid at its center. The algorithm minimizes the total distance from each movie to its centroid.

### Why it might beat the genre baseline

Unlike genre labels (which are coarse, overlapping, and subjective), K-Means uses all **269 features** simultaneously — capturing mood, setting, character dynamics, visual style, and more. This should find more nuanced groupings.

### Limitation

K-Means assumes clusters are roughly **spherical and equally sized**. If movies form elongated or irregular groupings, K-Means may split or merge them incorrectly. We'll test algorithms that don't have this limitation later.

### Choosing K

K-Means requires you to specify how many clusters to create. Two popular techniques help us pick:
- **Elbow Method**: Plot inertia vs K — look for where improvement slows down
- **Silhouette Score**: Measures how well each point fits its cluster vs neighbors (-1 to +1, higher is better)

In [ ]:
# Test a range of K values
K_range = range(2, 150)
inertias = []
silhouette_scores = []

print('Running K-Means for K = 2 through 15...')
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, labels, sample_size=5000, random_state=42)
    silhouette_scores.append(sil)
    print(f'  K={k:2d}  |  Inertia: {km.inertia_:>12,.0f}  |  Silhouette: {sil:.4f}')

print('\nDone!')

In [ ]:
# Plot both methods with rate-of-change subplots to reveal variability
# Use actual number of results collected (handles interrupted runs)
K_list = list(K_range)[:len(inertias)]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# --- Top-left: Elbow plot ---
axes[0, 0].plot(K_list, inertias, '-', color='steelblue', linewidth=1.5, alpha=0.9)
axes[0, 0].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[0, 0].set_ylabel('Inertia', fontsize=12)
axes[0, 0].set_title('Elbow Method — Inertia', fontsize=14)
axes[0, 0].xaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=15))

# --- Bottom-left: Inertia % change (highlights the elbow) ---
inertia_pct_change = [0] + [100 * (inertias[i] - inertias[i-1]) / inertias[i-1] for i in range(1, len(inertias))]
axes[1, 0].bar(K_list, inertia_pct_change, color='steelblue', alpha=0.7, width=0.8)
axes[1, 0].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[1, 0].set_ylabel('Inertia % Change', fontsize=12)
axes[1, 0].set_title('Inertia % Change (step-to-step)', fontsize=14)
axes[1, 0].xaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=15))
axes[1, 0].axhline(y=0, color='gray', linewidth=0.5)

# --- Top-right: Silhouette score ---
axes[0, 1].plot(K_list, silhouette_scores, '-', color='coral', linewidth=1.5, alpha=0.9)
best_idx = np.argmax(silhouette_scores)
axes[0, 1].scatter([K_list[best_idx]], [silhouette_scores[best_idx]],
                    color='red', s=100, zorder=5, edgecolors='black', linewidths=1.5,
                    label=f'Best: K={K_list[best_idx]} ({silhouette_scores[best_idx]:.4f})')
axes[0, 1].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[0, 1].set_ylabel('Silhouette Score', fontsize=12)
axes[0, 1].set_title('Silhouette Score', fontsize=14)
axes[0, 1].legend(fontsize=10)
axes[0, 1].xaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=15))

# --- Bottom-right: Silhouette with rolling average to show trend ---
window = max(3, len(K_list) // 20)  # adaptive window size
sil_rolling = pd.Series(silhouette_scores).rolling(window=window, center=True).mean()
axes[1, 1].plot(K_list, silhouette_scores, '-', color='coral', alpha=0.3, linewidth=1, label='Raw')
axes[1, 1].plot(K_list, sil_rolling, '-', color='darkred', linewidth=2.5, label=f'Rolling avg (window={window})')
axes[1, 1].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[1, 1].set_ylabel('Silhouette Score', fontsize=12)
axes[1, 1].set_title('Silhouette Score — Smoothed Trend', fontsize=14)
axes[1, 1].legend(fontsize=10)
axes[1, 1].xaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=15))

plt.tight_layout()
plt.show()

best_k_sil = K_list[best_idx]
print(f'\nHighest silhouette score at K = {best_k_sil} ({max(silhouette_scores):.4f})')
print('Look at the elbow plot too — pick the K where the curve bends.')
print('\nTIP: There\'s no single "right" answer. Pick a K that balances simplicity with good scores.')

### 5c — Pick your K

Based on the charts above, set `CHOSEN_K` below. A good starting point is wherever the silhouette score is highest or where the elbow plot bends. You can always come back and try a different K!

In [ ]:
# ==============================
# SET YOUR CHOSEN K HERE
# ==============================
CHOSEN_K = best_k_sil   # change this to any number you'd like to try

print(f'Using K = {CHOSEN_K} clusters')

---
## Step 6 — Run the Final K-Means Clustering

Now we run K-Means with our chosen K and assign every movie to a cluster.

In [ ]:
# Run K-Means
kmeans = KMeans(n_clusters=CHOSEN_K, random_state=42, n_init=20, max_iter=500)
cluster_labels = kmeans.fit_predict(X_scaled)

# Attach cluster labels back to our data
feature_matrix['cluster'] = cluster_labels

# Quick look at cluster sizes
print('Cluster sizes:')
print(feature_matrix['cluster'].value_counts().sort_index().to_string())

# Bar chart of cluster sizes
fig, ax = plt.subplots(figsize=(10, 4))
feature_matrix['cluster'].value_counts().sort_index().plot.bar(
    ax=ax, color=sns.color_palette('Set2', CHOSEN_K), edgecolor='white'
)
ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('Number of Movies', fontsize=12)
ax.set_title('Movies per Cluster', fontsize=14)
plt.tight_layout()
plt.show()

---
## Step 7 — Analyze Each Cluster

What makes each cluster unique? We'll look at the **top features** (highest average trigger value) for each cluster. This tells us the "personality" of each group.

In [ ]:
# Compute the average feature value for each cluster
feature_cols = [c for c in feature_matrix.columns if c != 'cluster']
cluster_profiles = feature_matrix.groupby('cluster')[feature_cols].mean()

# For each cluster, show the top 10 distinguishing features
TOP_N = 10
for cluster_id in range(CHOSEN_K):
    top_features = cluster_profiles.loc[cluster_id].nlargest(TOP_N)
    print(f'\n{"="*60}')
    print(f'CLUSTER {cluster_id}  ({(cluster_labels == cluster_id).sum():,} movies)')
    print(f'{"="*60}')
    for feat, val in top_features.items():
        bar = '█' * int(val * 5)
        print(f'  {feat:<45s} {val:.2f}  {bar}')

---
## Step 8 — Visualize K-Means Clusters with PCA

Using the PCA projection we computed earlier, let's see how K-Means divided the movie space. We'll also plot the **cluster centroids** — the center points K-Means is optimizing around.

In [ ]:
# Project the centroids into the same 2D PCA space
centroids_pca = pca.transform(kmeans.cluster_centers_)

# Plot movies + centroids
fig, ax = plt.subplots(figsize=(12, 8))
palette = sns.color_palette('Set2', CHOSEN_K)
for cluster_id in range(CHOSEN_K):
    mask = cluster_labels == cluster_id
    ax.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        c=[palette[cluster_id]], label=f'Cluster {cluster_id}',
        alpha=0.4, s=15, edgecolors='none'
    )

# Plot centroids as large bold markers
for cluster_id in range(CHOSEN_K):
    ax.scatter(
        centroids_pca[cluster_id, 0], centroids_pca[cluster_id, 1],
        c=[palette[cluster_id]], marker='X', s=300, edgecolors='black',
        linewidths=2, zorder=10
    )
    ax.annotate(
        f'  C{cluster_id}', (centroids_pca[cluster_id, 0], centroids_pca[cluster_id, 1]),
        fontsize=12, fontweight='bold', color='black'
    )

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
ax.set_title('K-Means Clusters — PCA Projection (X = centroid)', fontsize=14)
ax.legend(fontsize=10, markerscale=3)
plt.tight_layout()
plt.show()

---
## Step 9 — Visualize with t-SNE

**t-SNE** is another dimensionality reduction technique. It's better at preserving *local* structure (nearby neighbors) but can take a bit longer to run. It often reveals cluster shapes that PCA misses.

In [ ]:
# Safety imports — so this cell works even if you run it standalone
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# First reduce to 50 dims with PCA (speeds up t-SNE considerably)
pca_50 = PCA(n_components=50, random_state=42)
X_pca50 = pca_50.fit_transform(X_scaled)

# Run t-SNE on ALL 8,000 movies (full dataset, no sampling)
print(f'Running t-SNE on all {X_scaled.shape[0]:,} movies — this may take a minute...')
tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
X_tsne = tsne.fit_transform(X_pca50)

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
for cluster_id in range(CHOSEN_K):
    mask = cluster_labels == cluster_id
    ax.scatter(
        X_tsne[mask, 0], X_tsne[mask, 1],
        c=[palette[cluster_id]], label=f'Cluster {cluster_id}',
        alpha=0.5, s=15, edgecolors='none'
    )
ax.set_xlabel('t-SNE 1', fontsize=12)
ax.set_ylabel('t-SNE 2', fontsize=12)
ax.set_title(f'Movie Clusters — t-SNE Projection (all {X_scaled.shape[0]:,} movies)', fontsize=14)
ax.legend(fontsize=10, markerscale=3)
plt.tight_layout()
plt.show()
print('Done!')

---
## Step 10 — Genre Breakdown by Cluster

Let's see which genres ended up in which clusters. This is a great sanity check — if clustering worked well, we'd expect some genre patterns to emerge even though the algorithm never saw the genre labels.

In [ ]:
# Heatmap: top features most present in each cluster
feature_cols = [c for c in feature_matrix.columns if c != 'cluster']
cluster_profiles = feature_matrix.groupby('cluster')[feature_cols].mean()

# For each cluster, grab the top 15 features by average value
TOP_N = 15
top_feature_names = set()
for cluster_id in range(CHOSEN_K):
    top_feature_names.update(cluster_profiles.loc[cluster_id].nlargest(TOP_N).index.tolist())

# Build a DataFrame of just those features across all clusters
top_feature_names = sorted(top_feature_names)
heatmap_data = cluster_profiles[top_feature_names].T

# Sort features by max value across clusters for better visual grouping
heatmap_data = heatmap_data.loc[heatmap_data.max(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(14, max(8, len(heatmap_data) * 0.35)))
sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Average Trigger Value'})
ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('Feature', fontsize=12)
ax.set_title('Top Features Most Present in Each Cluster', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Create movie_clusters by merging cluster labels with genre data
movie_clusters = genres.rename(columns={'movie_id': 'imdb_id'}).merge(
    feature_matrix[['cluster']].reset_index(), on='imdb_id', how='inner'
)

# Create genre_rows by exploding multi-genre strings
genre_rows = movie_clusters.assign(
    genre_split=movie_clusters['genre'].str.split(', ')
).explode('genre_split')

# Heatmap: genre distribution across clusters
genre_rows_clean = genre_rows.reset_index(drop=True)

top_genres = genre_rows_clean['genre_split'].value_counts().head(12).index.tolist()
heatmap_data = pd.crosstab(
    genre_rows_clean['cluster'],
    genre_rows_clean['genre_split'],
    normalize='index'
)[top_genres]

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(heatmap_data, annot=True, fmt='.1%', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_xlabel('Genre', fontsize=12)
ax.set_ylabel('Cluster', fontsize=12)
ax.set_title('Genre Distribution Across Clusters (row-normalized)', fontsize=14)
plt.tight_layout()
plt.show()

---
## Step 11 — Browse Movies in Each Cluster

Let's look at some sample movies from each cluster to see if the groupings make intuitive sense.

In [ ]:
# Show 10 random movies from each cluster
for c in range(CHOSEN_K):
    sample = movie_clusters[movie_clusters['cluster'] == c].sample(
        min(10, len(movie_clusters[movie_clusters['cluster'] == c])),
        random_state=42
    )
    print(f'\nCluster {c} — sample movies:')
    for _, row in sample.iterrows():
        print(f'  • {row["movie_name"]}  [{row["genre"]}]')

---
## Step 12 — Export Results

Save the cluster assignments so you can use them later.

In [ ]:
# Create a clean export
export_df = movie_clusters[['imdb_id', 'movie_name', 'genre', 'cluster']].sort_values('cluster')
export_df.to_csv('movie_clusters_output.csv', index=False)

print(f'Saved {len(export_df):,} movies with cluster labels to movie_clusters_output.csv')
display(export_df.head(10))

---
## Step 9 — DBSCAN (Density-Based Clustering)

### Why try DBSCAN?

K-Means forces every movie into a cluster and assumes clusters are roughly spherical. But what if some movies are true outliers that don't belong anywhere? And what if the natural groupings are irregularly shaped?

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) addresses both issues:
- It finds clusters by looking for **dense regions** separated by sparse gaps
- Movies in low-density areas are labeled as **noise** (outliers) rather than forced into a cluster
- It can find **arbitrarily shaped** clusters that K-Means would split apart

### How it works

DBSCAN has two parameters:
- **`eps`** — the maximum distance two movies can be apart and still be "neighbors"
- **`min_samples`** — how many neighbors a point needs to be considered part of a dense region

The algorithm:
1. Pick an unvisited movie
2. Find all neighbors within `eps` distance
3. If it has ≥ `min_samples` neighbors, start a new cluster and expand outward
4. If not, mark it as noise (it may join a cluster later if a neighbor starts one)

### Finding a good `eps`
We'll use a **k-distance plot**: compute the distance from each point to its k-th nearest neighbor, sort those distances, and look for an "elbow". The elbow suggests a natural density threshold.

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN

# k-distance plot to find a good eps
# k = min_samples (we'll use 10 as a reasonable default for 8000 points)
k = 10
nn = NearestNeighbors(n_neighbors=k)
nn.fit(X_scaled)
distances, _ = nn.kneighbors(X_scaled)
k_distances = np.sort(distances[:, -1])  # distance to k-th neighbor

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(k_distances, color='steelblue', linewidth=1.5)
ax.set_xlabel('Points (sorted by distance)', fontsize=12)
ax.set_ylabel(f'Distance to {k}th nearest neighbor', fontsize=12)
ax.set_title(f'k-Distance Plot (k={k}) — look for the elbow', fontsize=14)

# Mark some candidate eps values for reference
for pct in [50, 75, 90]:
    val = np.percentile(k_distances, pct)
    ax.axhline(y=val, color='coral', linestyle='--', alpha=0.5)
    ax.annotate(f'  p{pct}={val:.3f}', xy=(len(k_distances)*0.85, val), fontsize=10, color='coral')

plt.tight_layout()
plt.show()

# Print percentiles to help pick eps
print('k-distance percentiles:')
for p in [25, 50, 75, 85, 90, 95]:
    print(f'  p{p:2d} = {np.percentile(k_distances, p):.4f}')

In [ ]:
# Run DBSCAN — try a few eps values based on the k-distance plot
# We use the 75th-90th percentile range as a starting point
eps_candidates = [
    np.percentile(k_distances, 60),
    np.percentile(k_distances, 75),
    np.percentile(k_distances, 85),
    np.percentile(k_distances, 90),
]

print('Testing DBSCAN with different eps values (min_samples=10):\n')
dbscan_results = {}
for eps_val in eps_candidates:
    db = DBSCAN(eps=eps_val, min_samples=10, n_jobs=-1)
    db_labels = db.fit_predict(X_scaled)
    n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
    n_noise = (db_labels == -1).sum()
    noise_pct = n_noise / len(db_labels) * 100

    sil = None
    if n_clusters >= 2 and n_clusters < len(X_scaled):
        # Only compute silhouette on non-noise points
        non_noise = db_labels != -1
        if non_noise.sum() > n_clusters:
            sil = silhouette_score(X_scaled[non_noise], db_labels[non_noise],
                                   sample_size=min(5000, non_noise.sum()), random_state=42)

    dbscan_results[eps_val] = {'labels': db_labels, 'n_clusters': n_clusters,
                                'n_noise': n_noise, 'silhouette': sil}
    sil_str = f'{sil:.4f}' if sil is not None else 'N/A'
    print(f'  eps={eps_val:.4f}  →  {n_clusters:3d} clusters, {n_noise:,} noise ({noise_pct:.1f}%),  silhouette: {sil_str}')

# Pick the best eps (highest silhouette among those with 2+ clusters)
valid = {k: v for k, v in dbscan_results.items() if v['silhouette'] is not None and v['n_clusters'] >= 2}
if valid:
    best_eps = max(valid, key=lambda k: valid[k]['silhouette'])
    dbscan_labels = dbscan_results[best_eps]['labels']
    print(f'\n→ Best: eps={best_eps:.4f} with {dbscan_results[best_eps]["n_clusters"]} clusters '
          f'(silhouette={dbscan_results[best_eps]["silhouette"]:.4f})')
else:
    # Fallback: pick the one with the most clusters
    best_eps = max(dbscan_results, key=lambda k: dbscan_results[k]['n_clusters'])
    dbscan_labels = dbscan_results[best_eps]['labels']
    print(f'\n→ Using eps={best_eps:.4f} ({dbscan_results[best_eps]["n_clusters"]} clusters)')

n_db_clusters = dbscan_results[best_eps]['n_clusters']

In [ ]:
# Visualize DBSCAN results on the PCA projection (reusing X_pca from Step 8)
fig, ax = plt.subplots(figsize=(12, 8))

# Plot noise points first (in gray)
noise_mask = dbscan_labels == -1
if noise_mask.any():
    ax.scatter(X_pca[noise_mask, 0], X_pca[noise_mask, 1],
               c='lightgray', alpha=0.3, s=8, label=f'Noise ({noise_mask.sum():,})')

# Plot each cluster
db_palette = sns.color_palette('tab10', n_db_clusters)
for cid in range(n_db_clusters):
    mask = dbscan_labels == cid
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=[db_palette[cid % len(db_palette)]], alpha=0.5, s=15,
               label=f'Cluster {cid} ({mask.sum():,})', edgecolors='none')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
ax.set_title(f'DBSCAN Clusters (eps={best_eps:.4f}) — PCA Projection', fontsize=14)
ax.legend(fontsize=9, markerscale=3, loc='best')
plt.tight_layout()
plt.show()

# Cluster size breakdown
print('DBSCAN cluster sizes:')
for cid in range(-1, n_db_clusters):
    count = (dbscan_labels == cid).sum()
    label = 'Noise' if cid == -1 else f'Cluster {cid}'
    print(f'  {label:<15s} {count:>5,} movies')

---
## Step 10 — Agglomerative (Hierarchical) Clustering

### Why try Agglomerative?

Both K-Means and DBSCAN require you to commit to parameters upfront. Agglomerative clustering gives you a **full hierarchy** of possible groupings, from "every movie in its own cluster" to "all movies in one cluster." You can then decide where to cut.

### How it works

Agglomerative clustering takes a **bottom-up** approach:
1. Start with every movie as its own cluster (8,000 clusters)
2. Find the two most similar clusters and merge them
3. Repeat until everything is in one giant cluster

The result is a **dendrogram** (tree diagram) showing the full merge history. You can "cut" the tree at any height to get different numbers of clusters.

We use **Ward linkage**, which merges clusters in a way that minimizes within-cluster variance (similar to K-Means' objective).

### Why it might find better clusters

Agglomerative doesn't assume spherical clusters and can capture hierarchical relationships — e.g., "thriller" movies might split into psychological thrillers and action thrillers at a finer cut.

**Note:** The dendrogram uses a 2,000-movie sample (for speed), but the actual clustering runs on all 8,000 movies.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering

# Build a dendrogram from a sample (full dataset would be very slow)
DENDRO_SAMPLE = 2000
np.random.seed(42)
dendro_idx = np.random.choice(X_scaled.shape[0], DENDRO_SAMPLE, replace=False)
X_dendro = X_scaled[dendro_idx]

print(f'Computing linkage on {DENDRO_SAMPLE:,} sampled movies...')
Z = linkage(X_dendro, method='ward')

# Plot the dendrogram (truncated to last 30 merges for readability)
fig, ax = plt.subplots(figsize=(14, 7))
dendrogram(
    Z,
    truncate_mode='lastp', p=30,    # show only last 30 merges
    leaf_rotation=90, leaf_font_size=9,
    ax=ax, color_threshold=0
)
ax.set_xlabel('Cluster size', fontsize=12)
ax.set_ylabel('Ward distance (merge cost)', fontsize=12)
ax.set_title('Agglomerative Dendrogram (sampled, last 30 merges)', fontsize=14)
ax.axhline(y=np.median(Z[-5:, 2]), color='coral', linestyle='--', alpha=0.7, label='Suggested cut')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Show the merge distances for the last few merges to help pick a cut
print('\nLast 10 merge distances (large jumps suggest natural cluster boundaries):')
for i in range(-10, 0):
    n_clusters_at_cut = -i
    print(f'  {n_clusters_at_cut:2d} clusters  →  merge distance: {Z[i, 2]:.2f}')

In [ ]:
# Run Agglomerative Clustering on the FULL dataset
# Use the same CHOSEN_K so we can compare fairly with K-Means
AGGLO_K = CHOSEN_K

print(f'Running Agglomerative Clustering with {AGGLO_K} clusters (ward linkage)...')
agglo = AgglomerativeClustering(n_clusters=AGGLO_K, linkage='ward')
agglo_labels = agglo.fit_predict(X_scaled)

agglo_sil = silhouette_score(X_scaled, agglo_labels, sample_size=5000, random_state=42)
print(f'Done! Silhouette score: {agglo_sil:.4f}\n')

# Cluster sizes
print('Agglomerative cluster sizes:')
for cid in range(AGGLO_K):
    count = (agglo_labels == cid).sum()
    print(f'  Cluster {cid}: {count:,} movies')

# PCA plot
fig, ax = plt.subplots(figsize=(12, 8))
agglo_palette = sns.color_palette('Set2', AGGLO_K)
for cid in range(AGGLO_K):
    mask = agglo_labels == cid
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=[agglo_palette[cid]], alpha=0.4, s=15,
               label=f'Cluster {cid} ({mask.sum():,})', edgecolors='none')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
ax.set_title(f'Agglomerative Clustering ({AGGLO_K} clusters, ward) — PCA Projection', fontsize=14)
ax.legend(fontsize=10, markerscale=3)
plt.tight_layout()
plt.show()

---
## Step 11 — Minkowski Distance Exploration

### Why try different distance metrics?

Every clustering algorithm relies on a **distance function** to decide which movies are "close" to each other. So far we've used Euclidean distance (optionally converted to cosine via L2 normalization). But what if a different way of measuring distance produces better clusters?

### How Minkowski distance works

**Minkowski distance** is a generalized distance formula controlled by a single parameter **p**:

$$d(x, y) = \left( \sum_{i=1}^{n} |x_i - y_i|^p \right)^{1/p}$$

| **p value** | **Name** | **What it measures** |
|:-----------:|:--------:|:---------------------|
| p = 1       | Manhattan (City-block) | Sum of absolute differences — treats every feature equally and linearly |
| p = 2       | Euclidean | Straight-line distance — penalizes large differences more (they get squared) |
| p → ∞       | Chebyshev | Maximum difference across any single feature |
| 0 < p < 1   | Sub-linear | Emphasizes small differences even more |

### Why p matters for movies

- **Low p (≈ 1):** Two movies are "far apart" if they differ *a little bit across many features*. Good for catching broad stylistic differences.
- **High p (≈ 2+):** Two movies are "far apart" if they differ *a lot on even one feature*. Good if a single strong feature (like "Suspenseful Atmosphere" = 3 vs 0) should dominate the grouping.

We'll cluster with Agglomerative Clustering (which accepts precomputed distance matrices) at several p values and see which produces the best silhouette score.

In [ ]:
from sklearn.metrics import pairwise_distances
from sklearn.cluster import AgglomerativeClustering

# ================================================================
# Explore different Minkowski p values
# ================================================================
# K-Means natively uses Euclidean (p=2). To cluster with other p
# values, we compute a Minkowski distance matrix and feed it to
# Agglomerative Clustering (which accepts precomputed distances).
# ================================================================

p_values = [0.5, 1, 1.5, 2, 3, 5]
USE_K = CHOSEN_K  # same K for fair comparison

results_mink = {}
print(f'Testing Minkowski distance with p = {p_values}  (K={USE_K} clusters)\n')

for p in p_values:
    print(f'  p = {p:<4} ...', end=' ', flush=True)
    
    # Compute pairwise Minkowski distance matrix
    dist_matrix = pairwise_distances(X_scaled, metric='minkowski', p=p)
    
    # Agglomerative with precomputed distances (average linkage works with any metric)
    agglo_mink = AgglomerativeClustering(
        n_clusters=USE_K,
        metric='precomputed',
        linkage='average'   # ward only works with Euclidean
    )
    labels_mink = agglo_mink.fit_predict(dist_matrix)
    
    # Silhouette score using the same Minkowski distance
    sil = silhouette_score(dist_matrix, labels_mink, metric='precomputed',
                           sample_size=5000, random_state=42)
    
    # Cluster sizes
    sizes = [int((labels_mink == c).sum()) for c in range(USE_K)]
    
    results_mink[p] = {'labels': labels_mink, 'silhouette': sil,
                       'sizes': sizes, 'dist_matrix': dist_matrix}
    print(f'silhouette = {sil:.4f}   sizes = {sizes}')

# Find best p
best_p = max(results_mink, key=lambda p: results_mink[p]['silhouette'])
print(f'\n→ Best silhouette at p = {best_p} ({results_mink[best_p]["silhouette"]:.4f})')

In [ ]:
# Visualize: silhouette score vs p, and PCA projections for each p

# Determine best_p from results (in case cell 49 was re-run partially)
best_p = max(results_mink, key=lambda p: results_mink[p]['silhouette'])

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for idx, p in enumerate(p_values):
    ax = axes[idx // 3][idx % 3]
    labels_p = results_mink[p]['labels']
    sil_p = results_mink[p]['silhouette']
    mink_palette = sns.color_palette('Set2', USE_K)
    
    for cid in range(USE_K):
        mask = labels_p == cid
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=[mink_palette[cid]], alpha=0.3, s=10, edgecolors='none',
                   label=f'C{cid} ({mask.sum():,})')
    
    ax.set_title(f'p = {p}  (sil={sil_p:.4f})', fontsize=12,
                 fontweight='bold' if p == best_p else 'normal',
                 color='darkred' if p == best_p else 'black')
    ax.set_xlabel('PC1', fontsize=9)
    ax.set_ylabel('PC2', fontsize=9)
    ax.legend(fontsize=7, markerscale=2, loc='best')
    
    if p == best_p:
        ax.set_facecolor('#fff8f0')

plt.suptitle(f'Minkowski Distance — Clustering at Different p Values (K={USE_K})',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Bar chart: silhouette score vs p
fig, ax = plt.subplots(figsize=(10, 5))
sil_values = [results_mink[p]['silhouette'] for p in p_values]
colors = ['darkred' if p == best_p else 'steelblue' for p in p_values]
bars = ax.bar([str(p) for p in p_values], sil_values, color=colors, edgecolor='white', width=0.6)
ax.set_xlabel('Minkowski p', fontsize=12)
ax.set_ylabel('Silhouette Score', fontsize=12)
ax.set_title('Silhouette Score by Minkowski p Value', fontsize=14)
for bar, val in zip(bars, sil_values):
    ax.annotate(f'{val:.4f}', (bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary interpretation
print('\n' + '='*60)
print('MINKOWSKI DISTANCE SUMMARY')
print('='*60)
print(f'{"p":<8s} {"Name":<20s} {"Silhouette":>12s} {"Cluster Sizes":>30s}')
print(f'{"-"*8} {"-"*20} {"-"*12} {"-"*30}')
names = {0.5: 'Sub-linear', 1: 'Manhattan', 1.5: 'Intermediate', 2: 'Euclidean', 3: 'Cubic', 5: 'Near-Chebyshev'}
for p in p_values:
    r = results_mink[p]
    marker = ' ← best' if p == best_p else ''
    print(f'{p:<8} {names.get(p, ""):<20s} {r["silhouette"]:>12.4f} {str(r["sizes"]):>30s}{marker}')

print(f'\nInterpretation:')
print(f'  Lower p (→ Manhattan) treats all feature differences equally.')
print(f'  Higher p (→ Chebyshev) lets the biggest single difference dominate.')
print(f'  For this movie data, p = {best_p} gives the best cluster separation.')

---
# Part II — Taxonomy-Based Advanced Clustering

**Why go beyond basic K-Means?**

So far we treated all 248 features as a flat list. But our `feature_taxonomy_categorized.csv` organizes those features into a rich **hierarchy of categories** — Setting, Category 11, Relationships, Category 07, Category 20, and many more.

This structure is valuable information that basic K-Means throws away. The four advanced methods below each exploit the taxonomy in a different way:

| Method | Core idea | What the taxonomy provides |
|:-------|:----------|:---------------------------|
| **Multi-View Clustering** | Cluster movies separately within each category, then fuse the results | Each category = one "view" of the data |
| **Hierarchical + Category Metrics** | Use different distance metrics for different types of features | Category labels tell us *which* metric fits each feature group |
| **Ensemble Clustering** | Run many clusterings and let them vote | Category-aware base clusterers produce diverse perspectives |
| **Subspace Clustering** | Find clusters that only exist in specific feature subsets | Categories define meaningful subspaces to search |

Let's load the taxonomy and get started.

---
## Step 17 — Load the Feature Taxonomy

The taxonomy CSV maps every feature to a **Category** (broad grouping like "Category 13") and **Subcategory** (finer grouping like "Region / Country"). We'll use the Category level to define our clustering views.

In [ ]:
# Load the categorized taxonomy
taxonomy_cat = pd.read_csv('feature_taxonomy_categorized.csv')
print(f'Taxonomy shape: {taxonomy_cat.shape}')
display(taxonomy_cat.head(10))

# Build a mapping: feature name → category
feature_to_category = dict(zip(taxonomy_cat['Feature'], taxonomy_cat['Category']))
feature_to_subcategory = dict(zip(taxonomy_cat['Feature'], taxonomy_cat['Subcategory']))
feature_to_type = dict(zip(taxonomy_cat['Feature'], taxonomy_cat['Feature Type']))

# Which categories do we have and how many features in each?
cat_counts = taxonomy_cat['Category'].value_counts()
print(f'\n{len(cat_counts)} categories found:\n')
for cat, count in cat_counts.items():
    print(f'  {cat:<45s} {count:>3d} features')

In [ ]:
# Group feature columns by category
# Only include features that exist in BOTH the taxonomy and our feature matrix
feature_cols = [c for c in feature_matrix.columns if c != 'cluster' and not c.startswith('genre_')]

category_features = {}  # category → list of column names
unmapped = []

for col in feature_cols:
    cat = feature_to_category.get(col)
    if cat:
        category_features.setdefault(cat, []).append(col)
    else:
        unmapped.append(col)

print(f'Mapped {sum(len(v) for v in category_features.values())} features across {len(category_features)} categories')
if unmapped:
    print(f'Unmapped features (will be grouped as "Other"): {len(unmapped)}')
    # Group unmapped features under "Other"
    if unmapped:
        category_features['Other'] = unmapped

# Also add genre columns as their own view
genre_cols = [c for c in feature_matrix.columns if c.startswith('genre_')]
if genre_cols:
    category_features['Genre (one-hot)'] = genre_cols
    print(f'Added {len(genre_cols)} genre columns as a separate view')

print(f'\nFinal view breakdown:')
for cat, feats in sorted(category_features.items(), key=lambda x: -len(x[1])):
    print(f'  {cat:<45s} {len(feats):>3d} features')

---
## Step 18 — Multi-View Clustering

### What is Multi-View Clustering?

Imagine you're describing a movie to different experts:
- A **cinematographer** focuses on visual style, camera work, and color
- A **screenwriter** focuses on dialogue quality, plot structure, and character arcs
- A **psychologist** focuses on emotional tone, protagonist psychology, and relationships

Each expert sees the same movie through a different **lens** (or *view*). Multi-View Clustering formalizes this: we cluster movies independently within each category view, then **fuse** the results.

### Our approach: Late Fusion via Co-Association Matrix

1. **Cluster each view independently** — Run K-Means on just the "Category 07" features, just the "Setting" features, etc.
2. **Build a co-association matrix** — For each pair of movies, count how often they end up in the *same* cluster across all views. Movies that consistently cluster together (regardless of which lens we use) are truly similar.
3. **Final clustering** — Treat the co-association matrix as a similarity matrix and run Spectral Clustering to get the final groups.

**Why is this better than flat clustering?**
- Each category contributes equally, regardless of how many features it has (a category with 3 features gets the same vote as one with 30)
- Movies that are similar across *multiple* dimensions (not just one dominant category) get grouped together
- We can see *which views* agree and which disagree — revealing interesting structure

In [ ]:
from sklearn.cluster import SpectralClustering

# =============================================
# MULTI-VIEW CLUSTERING — Late Fusion
# =============================================
N_CLUSTERS_PER_VIEW = 8   # clusters within each view
N_FINAL_CLUSTERS = CHOSEN_K if CHOSEN_K > 2 else 6  # final output clusters

# Drop the 'cluster' column temporarily to avoid issues
feature_data = feature_matrix.drop(columns=['cluster'], errors='ignore')

print(f'Multi-View Clustering: {len(category_features)} views, '
      f'{N_CLUSTERS_PER_VIEW} clusters per view, {N_FINAL_CLUSTERS} final clusters\n')

# Step 1: Cluster each view independently
view_labels = {}  # view_name → array of cluster labels
view_silhouettes = {}

for view_name, view_cols in category_features.items():
    # Extract and scale this view's features
    X_view = feature_data[view_cols].values
    X_view_scaled = StandardScaler().fit_transform(X_view)
    X_view_scaled = normalize(X_view_scaled, norm='l2')

    # Cluster
    n_clust = min(N_CLUSTERS_PER_VIEW, len(view_cols))  # can't have more clusters than features
    if n_clust < 2:
        n_clust = 2
    km_view = KMeans(n_clusters=n_clust, random_state=42, n_init=10)
    labels = km_view.fit_predict(X_view_scaled)
    view_labels[view_name] = labels

    sil = silhouette_score(X_view_scaled, labels, sample_size=min(3000, len(labels)), random_state=42)
    view_silhouettes[view_name] = sil
    print(f'  {view_name:<45s} {len(view_cols):>3d} feats  →  sil={sil:.4f}')

print(f'\nAll {len(view_labels)} views clustered.')

In [ ]:
# Step 2: Build the Co-Association Matrix
# For each view, create a binary "same cluster" matrix and average across all views
print('Building co-association matrix (this may take a moment)...')

n_movies = len(feature_data)
co_assoc = np.zeros((n_movies, n_movies), dtype=np.float32)

for view_name, labels in view_labels.items():
    # Efficiently build same-cluster matrix using label comparison
    # For each pair (i, j): 1 if labels[i] == labels[j], else 0
    for cluster_id in np.unique(labels):
        members = np.where(labels == cluster_id)[0]
        # Add 1 to all pairs within this cluster
        co_assoc[np.ix_(members, members)] += 1.0

# Normalize by number of views
co_assoc /= len(view_labels)

print(f'Co-association matrix shape: {co_assoc.shape}')
print(f'Value range: [{co_assoc.min():.3f}, {co_assoc.max():.3f}]')
print(f'Mean co-association: {co_assoc.mean():.4f}')
print(f'  → Two random movies land in the same cluster {co_assoc.mean()*100:.1f}% of the time on average')

In [ ]:
# Step 3: Final clustering from co-association matrix using Spectral Clustering
print(f'Running Spectral Clustering with K={N_FINAL_CLUSTERS} on co-association matrix...')

spectral = SpectralClustering(
    n_clusters=N_FINAL_CLUSTERS,
    affinity='precomputed',
    random_state=42,
    n_init=10
)
multiview_labels = spectral.fit_predict(co_assoc)

# Evaluate — silhouette_score with metric='precomputed' expects a DISTANCE matrix
# (0 on diagonal, higher = farther apart), so convert similarity → distance
co_assoc_dist = 1.0 - co_assoc
np.fill_diagonal(co_assoc_dist, 0)

mv_sil = silhouette_score(co_assoc_dist, multiview_labels, metric='precomputed',
                           sample_size=5000, random_state=42)
print(f'Multi-View Clustering silhouette: {mv_sil:.4f}')
print(f'\nCluster sizes:')
for cid in range(N_FINAL_CLUSTERS):
    print(f'  Cluster {cid}: {(multiview_labels == cid).sum():,} movies')

In [ ]:
# =============================================
# MULTI-VIEW VISUALIZATIONS
# =============================================

# 1. Per-view silhouette scores (shows which views produce the cleanest clusters)
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Bar chart of per-view silhouette scores
view_names = list(view_silhouettes.keys())
view_sils = [view_silhouettes[v] for v in view_names]
sort_idx = np.argsort(view_sils)[::-1]
sorted_names = [view_names[i] for i in sort_idx]
sorted_sils = [view_sils[i] for i in sort_idx]
n_feats_per_view = [len(category_features[v]) for v in sorted_names]

colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(sorted_names)))
bars = axes[0].barh(range(len(sorted_names)), sorted_sils, color=colors, edgecolor='white')
axes[0].set_yticks(range(len(sorted_names)))
axes[0].set_yticklabels([f'{n} ({nf}f)' for n, nf in zip(sorted_names, n_feats_per_view)], fontsize=9)
axes[0].set_xlabel('Silhouette Score', fontsize=11)
axes[0].set_title('Per-View Clustering Quality', fontsize=13)
axes[0].invert_yaxis()
for bar, val in zip(bars, sorted_sils):
    axes[0].text(val + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.3f}',
                va='center', fontsize=8)

# 2. Co-association matrix heatmap (sampled for visibility)
sample_size = 200
np.random.seed(42)
sample_idx = np.sort(np.random.choice(n_movies, sample_size, replace=False))
# Sort sample by multi-view cluster for block-diagonal structure
sample_order = np.argsort(multiview_labels[sample_idx])
ordered_idx = sample_idx[sample_order]

co_sample = co_assoc[np.ix_(ordered_idx, ordered_idx)]
im = axes[1].imshow(co_sample, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
plt.colorbar(im, ax=axes[1], label='Co-association score')
axes[1].set_title(f'Co-Association Matrix ({sample_size} movies, sorted by cluster)', fontsize=13)
axes[1].set_xlabel('Movie index')
axes[1].set_ylabel('Movie index')

plt.tight_layout()
plt.show()

In [ ]:
# 3. PCA projection of multi-view clusters
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# PCA view
mv_palette = sns.color_palette('Set2', N_FINAL_CLUSTERS)
for cid in range(N_FINAL_CLUSTERS):
    mask = multiview_labels == cid
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=[mv_palette[cid]], alpha=0.4, s=12, edgecolors='none',
                    label=f'Cluster {cid} ({mask.sum():,})')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=11)
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=11)
axes[0].set_title('Multi-View Clustering — PCA Projection', fontsize=13)
axes[0].legend(fontsize=8, markerscale=3)

# t-SNE view (reusing X_tsne from earlier)
for cid in range(N_FINAL_CLUSTERS):
    mask = multiview_labels == cid
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                    c=[mv_palette[cid]], alpha=0.4, s=12, edgecolors='none',
                    label=f'Cluster {cid} ({mask.sum():,})')
axes[1].set_xlabel('t-SNE 1', fontsize=11)
axes[1].set_ylabel('t-SNE 2', fontsize=11)
axes[1].set_title('Multi-View Clustering — t-SNE Projection', fontsize=13)
axes[1].legend(fontsize=8, markerscale=3)

plt.tight_layout()
plt.show()

# 4. View agreement analysis — which views contribute most to the final clusters?
print('\nView Agreement Analysis:')
print('For each view, how well does its clustering align with the final multi-view result?\n')
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

agreement_data = []
for view_name, labels in view_labels.items():
    ari = adjusted_rand_score(multiview_labels, labels)
    nmi = normalized_mutual_info_score(multiview_labels, labels)
    agreement_data.append({'View': view_name, 'ARI': ari, 'NMI': nmi})

agreement_df = pd.DataFrame(agreement_data).sort_values('NMI', ascending=False)
print(f'{"View":<45s} {"ARI":>8s} {"NMI":>8s}')
print('-' * 63)
for _, row in agreement_df.iterrows():
    print(f'{row["View"]:<45s} {row["ARI"]:>8.4f} {row["NMI"]:>8.4f}')

---
## Step 19 — Hierarchical Clustering with Category-Specific Metrics

### The Problem with One-Size-Fits-All Distance

When we ran Agglomerative Clustering in Step 14, we used Ward linkage with Euclidean distance for *all* features. But different categories of features have fundamentally different characteristics:

- **Setting features** (location, time period) are mostly binary/sparse — a movie is either "Feature 0233" or it isn't. **Jaccard distance** (overlap of non-zero features) is natural here.
- **Category 07 features** are continuous intensity scores on a scale. **Cosine distance** (pattern similarity) captures mood profiles well.
- **Category 20 features** describe technical craft. **Euclidean distance** works well when you care about magnitude differences.

### Our approach: Weighted Distance Fusion

1. **Compute a separate distance matrix** for each category using the metric that best fits that category's feature type
2. **Normalize** each matrix to [0, 1] so no single category dominates
3. **Average** the normalized matrices (optionally with category weights)
4. **Run Hierarchical Clustering** on the fused distance matrix

This way, "Category 13" uses Jaccard distance while "Category 07" uses cosine distance — each feature group is measured on its own terms.

In [ ]:
from sklearn.metrics import pairwise_distances
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

# =============================================
# CATEGORY-SPECIFIC METRIC ASSIGNMENT
# =============================================
# Assign a distance metric to each category based on the feature type
# Objective/binary-ish features → Jaccard; Subjective/continuous → Cosine; Mixed → Euclidean

def assign_metric(category_name, feature_names):
    """Choose the best distance metric for a feature category."""
    # Check the feature types from taxonomy
    types = [feature_to_type.get(f, 'Hybrid') for f in feature_names
             if not f.startswith('genre_')]

    # Count types
    n_obj = sum(1 for t in types if t == 'Objective')
    n_subj = sum(1 for t in types if t == 'Subjective')
    n_hybrid = sum(1 for t in types if t == 'Hybrid')

    # Genre columns are binary → Jaccard
    if category_name == 'Genre (one-hot)':
        return 'jaccard'
    # Mostly objective/binary features → Jaccard
    if n_obj > n_subj + n_hybrid:
        return 'jaccard'
    # Mostly subjective/continuous → Cosine
    if n_subj > n_obj + n_hybrid:
        return 'cosine'
    # Default for mixed → Euclidean
    return 'euclidean'

# Assign metrics
category_metrics = {}
for cat, feats in category_features.items():
    metric = assign_metric(cat, feats)
    category_metrics[cat] = metric

print(f'{"Category":<45s} {"Metric":<12s} {"# Features":>10s}')
print('=' * 70)
for cat in sorted(category_metrics.keys()):
    print(f'{cat:<45s} {category_metrics[cat]:<12s} {len(category_features[cat]):>10d}')

In [ ]:
# =============================================
# COMPUTE CATEGORY-SPECIFIC DISTANCE MATRICES & FUSE
# =============================================
print('Computing per-category distance matrices...')

# Toggle to allow distance fusion on raw or standardized per-category features
SCALE_NUMERIC_VIEWS = True

n = len(feature_data)
fused_distance = np.zeros((n, n), dtype=np.float64)
n_views_counted = 0

for cat, feats in category_features.items():
    metric = category_metrics[cat]
    X_cat = feature_data[feats].values.astype(np.float64)

    # Scale numeric views if enabled
    if SCALE_NUMERIC_VIEWS:
        X_cat_num = StandardScaler().fit_transform(X_cat)
    else:
        X_cat_num = X_cat

    # Handle Jaccard: it needs binary input
    if metric == 'jaccard':
        # Binarize: anything > 0 becomes 1
        X_cat_binary = (X_cat > 0).astype(float)
        # Jaccard distance = 1 - Jaccard similarity
        dist = pairwise_distances(X_cat_binary, metric='jaccard')
    elif metric == 'cosine':
        dist = pairwise_distances(X_cat_num, metric='cosine')
    else:  # euclidean
        dist = pairwise_distances(X_cat_num, metric='euclidean')

    # Handle NaN/inf (can happen with sparse or all-zero vectors)
    dist = np.nan_to_num(dist, nan=0.0, posinf=1.0, neginf=0.0)

    # Normalize to [0, 1]
    d_max = dist.max()
    if d_max > 0:
        dist = dist / d_max

    # Enforce distance-matrix validity for downstream scipy/sklearn consumers
    dist = 0.5 * (dist + dist.T)
    np.fill_diagonal(dist, 0.0)

    fused_distance += dist
    n_views_counted += 1
    print(f'  {cat:<45s} {metric:<10s} range=[{dist.min():.3f}, {dist.max():.3f}]')

# Average and re-enforce validity after floating-point accumulation
fused_distance /= n_views_counted
fused_distance = 0.5 * (fused_distance + fused_distance.T)
np.fill_diagonal(fused_distance, 0.0)
fused_distance = np.clip(fused_distance, 0.0, None)

print('Numeric-view scaling:', 'ON (z-score per category)' if SCALE_NUMERIC_VIEWS else 'OFF (raw per category)')
print(f'\nFused distance matrix: shape={fused_distance.shape}, '
      f'range=[{fused_distance.min():.4f}, {fused_distance.max():.4f}]')

In [ ]:
# =============================================
# HIERARCHICAL CLUSTERING ON FUSED DISTANCE
# =============================================
HIER_K = N_FINAL_CLUSTERS  # same K for fair comparison

# Dendrogram on a sample (full 8000×8000 linkage is expensive)
SAMPLE_N = 2000
np.random.seed(42)
sample_idx = np.random.choice(n, SAMPLE_N, replace=False)
dist_sample = fused_distance[np.ix_(sample_idx, sample_idx)]

# Defensively enforce valid distance-matrix structure before squareform
# (floating-point accumulation can introduce tiny asymmetries)
dist_sample = 0.5 * (dist_sample + dist_sample.T)
np.fill_diagonal(dist_sample, 0.0)

# Convert to condensed form for scipy
from scipy.spatial.distance import squareform
dist_condensed = squareform(dist_sample)

print(f'Computing linkage on {SAMPLE_N} sampled movies...')
Z_cat = linkage(dist_condensed, method='average')

# Plot dendrogram
fig, ax = plt.subplots(figsize=(16, 7))
dendrogram(Z_cat, truncate_mode='lastp', p=40, leaf_rotation=90,
           leaf_font_size=8, ax=ax, color_threshold=0)
ax.set_xlabel('Cluster size', fontsize=12)
ax.set_ylabel('Fused distance (merge cost)', fontsize=12)
ax.set_title('Hierarchical Dendrogram — Category-Specific Metrics (sampled)', fontsize=14)

# Show last merge distances
print('\nLast 10 merge distances:')
for i in range(-10, 0):
    print(f'  {-i:2d} clusters → merge distance: {Z_cat[i, 2]:.4f}')

plt.tight_layout()
plt.show()

In [ ]:
# Run Agglomerative on full dataset with fused distance
print(f'Running Agglomerative Clustering (K={HIER_K}) on fused distance matrix...')
hier_cat = AgglomerativeClustering(
    n_clusters=HIER_K,
    metric='precomputed',
    linkage='average'
)
hier_cat_labels = hier_cat.fit_predict(fused_distance)

hier_cat_sil = silhouette_score(fused_distance, hier_cat_labels, metric='precomputed',
                                 sample_size=5000, random_state=42)
print(f'Silhouette score: {hier_cat_sil:.4f}')
print(f'\nCluster sizes:')
for cid in range(HIER_K):
    print(f'  Cluster {cid}: {(hier_cat_labels == cid).sum():,} movies')

# PCA + t-SNE visualization
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
hier_palette = sns.color_palette('tab10', HIER_K)

for cid in range(HIER_K):
    mask = hier_cat_labels == cid
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=[hier_palette[cid]], alpha=0.4, s=12, edgecolors='none',
                    label=f'Cluster {cid} ({mask.sum():,})')
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                    c=[hier_palette[cid]], alpha=0.4, s=12, edgecolors='none',
                    label=f'Cluster {cid} ({mask.sum():,})')

axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=11)
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=11)
axes[0].set_title('Hierarchical (Category Metrics) — PCA', fontsize=13)
axes[0].legend(fontsize=8, markerscale=3)

axes[1].set_xlabel('t-SNE 1', fontsize=11)
axes[1].set_ylabel('t-SNE 2', fontsize=11)
axes[1].set_title('Hierarchical (Category Metrics) — t-SNE', fontsize=13)
axes[1].legend(fontsize=8, markerscale=3)

plt.tight_layout()
plt.show()

In [ ]:
# Which category metrics drive the most cluster separation?
# Compare: for each category, how well does its distance alone predict the final clusters?
print('Category Contribution Analysis:')
print('How well does each category\'s distance matrix alone separate the final clusters?\n')

cat_contrib = []
for cat, feats in category_features.items():
    metric = category_metrics[cat]
    X_cat = feature_data[feats].values
    X_cat_scaled = StandardScaler().fit_transform(X_cat)

    if metric == 'jaccard':
        X_binary = (feature_data[feats].values > 0).astype(float)
        dist_cat = pairwise_distances(X_binary, metric='jaccard')
    elif metric == 'cosine':
        dist_cat = pairwise_distances(X_cat_scaled, metric='cosine')
    else:
        dist_cat = pairwise_distances(X_cat_scaled, metric='euclidean')

    dist_cat = np.nan_to_num(dist_cat, nan=0.0)
    d_max = dist_cat.max()
    if d_max > 0:
        dist_cat /= d_max

    sil_cat = silhouette_score(dist_cat, hier_cat_labels, metric='precomputed',
                                sample_size=min(3000, n), random_state=42)
    cat_contrib.append({'Category': cat, 'Metric': metric, 'Silhouette': sil_cat, 'N_Features': len(feats)})

contrib_df = pd.DataFrame(cat_contrib).sort_values('Silhouette', ascending=False)

fig, ax = plt.subplots(figsize=(12, 7))
colors = ['#e74c3c' if m == 'jaccard' else '#3498db' if m == 'cosine' else '#2ecc71'
          for m in contrib_df['Metric']]
bars = ax.barh(range(len(contrib_df)), contrib_df['Silhouette'], color=colors, edgecolor='white')
ax.set_yticks(range(len(contrib_df)))
ax.set_yticklabels([f"{row['Category']} [{row['Metric']}]" for _, row in contrib_df.iterrows()], fontsize=9)
ax.set_xlabel('Silhouette Score (using final cluster labels)', fontsize=11)
ax.set_title('Category Contribution to Hierarchical Clustering', fontsize=13)
ax.invert_yaxis()

# Legend for metric colors
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#e74c3c', label='Jaccard'),
                   Patch(facecolor='#3498db', label='Cosine'),
                   Patch(facecolor='#2ecc71', label='Euclidean')]
ax.legend(handles=legend_elements, title='Distance Metric', fontsize=9)

plt.tight_layout()
plt.show()

---
## Step 20 — Ensemble Clustering

### Why Ensemble?

A single clustering algorithm makes one set of assumptions about the data structure. **Ensemble Clustering** is like taking a vote: run many *different* clusterings (different algorithms, parameters, feature subsets) and combine them into a single consensus result.

Think of it like this: if K-Means with K=5, K-Means with K=10, Agglomerative with Ward, AND Agglomerative with Average all agree that movies A and B belong together — that's strong evidence they're truly similar.

### Our approach: Category-Aware Ensemble

We build a diverse ensemble by varying three axes:
1. **Algorithm** — K-Means vs. Agglomerative
2. **K value** — Different numbers of clusters
3. **Feature subset** — Each category provides a different base clustering

Then we build a **co-association matrix** (like Multi-View, but with more diverse inputs) and extract final clusters via Spectral Clustering.

### How is this different from Multi-View?
- **Multi-View** uses one clustering per category view, all with the same algorithm and K
- **Ensemble** uses many clusterings with different algorithms, K values, AND feature subsets
- Ensemble is more robust because it averages over more sources of variation

In [ ]:
# =============================================
# ENSEMBLE CLUSTERING — Build diverse base clusterings
# =============================================
from sklearn.cluster import AgglomerativeClustering

ensemble_labels_list = []
ensemble_descriptions = []

# --- Component 1: K-Means at different K values on full feature set ---
for k in [4, 6, 8, 10, 12, 15]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    ensemble_labels_list.append(labels)
    ensemble_descriptions.append(f'KMeans(K={k}, full)')

# --- Component 2: Agglomerative with different linkages ---
for link in ['ward', 'average', 'complete']:
    for k in [6, 10]:
        if link == 'ward':
            agglo = AgglomerativeClustering(n_clusters=k, linkage=link)
            labels = agglo.fit_predict(X_scaled)
        else:
            # Non-ward linkages work with precomputed distances too
            agglo = AgglomerativeClustering(n_clusters=k, linkage=link)
            labels = agglo.fit_predict(X_scaled)
        ensemble_labels_list.append(labels)
        ensemble_descriptions.append(f'Agglo({link}, K={k})')

# --- Component 3: K-Means per category view (reuse from multi-view) ---
for view_name, labels in view_labels.items():
    ensemble_labels_list.append(labels)
    ensemble_descriptions.append(f'KMeans(view={view_name[:25]})')

print(f'Ensemble contains {len(ensemble_labels_list)} base clusterings:\n')
for i, desc in enumerate(ensemble_descriptions):
    n_clust = len(np.unique(ensemble_labels_list[i]))
    print(f'  [{i+1:2d}] {desc:<50s} ({n_clust} clusters)')

In [ ]:
# =============================================
# BUILD ENSEMBLE CO-ASSOCIATION MATRIX
# =============================================
print(f'Building co-association matrix from {len(ensemble_labels_list)} base clusterings...')

n = len(feature_data)
ensemble_co_assoc = np.zeros((n, n), dtype=np.float32)

for labels in ensemble_labels_list:
    for cid in np.unique(labels):
        members = np.where(labels == cid)[0]
        ensemble_co_assoc[np.ix_(members, members)] += 1.0

ensemble_co_assoc /= len(ensemble_labels_list)

print(f'Co-association matrix: shape={ensemble_co_assoc.shape}')
print(f'Mean co-association: {ensemble_co_assoc.mean():.4f}')
print(f'Value range: [{ensemble_co_assoc.min():.3f}, {ensemble_co_assoc.max():.3f}]')

In [ ]:
# =============================================
# FINAL ENSEMBLE CLUSTERING
# =============================================
ENSEMBLE_K = N_FINAL_CLUSTERS

print(f'Running Spectral Clustering (K={ENSEMBLE_K}) on ensemble co-association matrix...')
spectral_ens = SpectralClustering(
    n_clusters=ENSEMBLE_K,
    affinity='precomputed',
    random_state=42,
    n_init=10
)
ensemble_labels = spectral_ens.fit_predict(ensemble_co_assoc)

ensemble_co_dist = 1.0 - ensemble_co_assoc
np.fill_diagonal(ensemble_co_dist, 0.0)
ens_sil = silhouette_score(ensemble_co_dist, ensemble_labels, metric='precomputed',
                            sample_size=5000, random_state=42)
print(f'Ensemble Clustering silhouette: {ens_sil:.4f}')
print(f'\nCluster sizes:')
for cid in range(ENSEMBLE_K):
    print(f'  Cluster {cid}: {(ensemble_labels == cid).sum():,} movies')

In [ ]:
# =============================================
# ENSEMBLE VISUALIZATIONS
# =============================================
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# 1. PCA projection
ens_palette = sns.color_palette('Set2', ENSEMBLE_K)
for cid in range(ENSEMBLE_K):
    mask = ensemble_labels == cid
    axes[0, 0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                        c=[ens_palette[cid]], alpha=0.4, s=12, edgecolors='none',
                        label=f'Cluster {cid} ({mask.sum():,})')
axes[0, 0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=11)
axes[0, 0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=11)
axes[0, 0].set_title('Ensemble Clustering — PCA Projection', fontsize=13)
axes[0, 0].legend(fontsize=8, markerscale=3)

# 2. t-SNE projection
for cid in range(ENSEMBLE_K):
    mask = ensemble_labels == cid
    axes[0, 1].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                        c=[ens_palette[cid]], alpha=0.4, s=12, edgecolors='none',
                        label=f'Cluster {cid} ({mask.sum():,})')
axes[0, 1].set_xlabel('t-SNE 1', fontsize=11)
axes[0, 1].set_ylabel('t-SNE 2', fontsize=11)
axes[0, 1].set_title('Ensemble Clustering — t-SNE Projection', fontsize=13)
axes[0, 1].legend(fontsize=8, markerscale=3)

# 3. Co-association matrix heatmap (sampled)
sample_size = 300
np.random.seed(42)
sample_idx = np.sort(np.random.choice(n, sample_size, replace=False))
sample_order = np.argsort(ensemble_labels[sample_idx])
ordered_idx = sample_idx[sample_order]
co_sample = ensemble_co_assoc[np.ix_(ordered_idx, ordered_idx)]

im = axes[1, 0].imshow(co_sample, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
plt.colorbar(im, ax=axes[1, 0], label='Co-association score')
axes[1, 0].set_title(f'Ensemble Co-Association Matrix ({sample_size} movies)', fontsize=13)

# 4. Consensus strength distribution
# For each movie, compute its average co-association with its own cluster members
consensus_strength = []
for i in range(n):
    own_cluster = ensemble_labels[i]
    members = np.where(ensemble_labels == own_cluster)[0]
    # Sample for speed
    if len(members) > 200:
        sampled = np.random.choice(members, 200, replace=False)
    else:
        sampled = members
    avg_co = ensemble_co_assoc[i, sampled].mean()
    consensus_strength.append(avg_co)

consensus_strength = np.array(consensus_strength)
axes[1, 1].hist(consensus_strength, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[1, 1].axvline(x=np.median(consensus_strength), color='red', linestyle='--',
                     label=f'Median: {np.median(consensus_strength):.3f}')
axes[1, 1].set_xlabel('Average co-association with own cluster', fontsize=11)
axes[1, 1].set_ylabel('Number of movies', fontsize=11)
axes[1, 1].set_title('Consensus Strength Distribution', fontsize=13)
axes[1, 1].legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f'\nConsensus strength stats:')
print(f'  Mean:   {consensus_strength.mean():.4f}')
print(f'  Median: {np.median(consensus_strength):.4f}')
print(f'  Min:    {consensus_strength.min():.4f} (weakest cluster member)')
print(f'  Max:    {consensus_strength.max():.4f} (strongest cluster member)')

In [ ]:
# Ensemble diversity: how much do the base clusterings agree with each other?
print('Base Clustering Agreement Matrix (ARI between all pairs):\n')

n_base = len(ensemble_labels_list)
ari_matrix = np.zeros((n_base, n_base))
for i in range(n_base):
    for j in range(i, n_base):
        ari = adjusted_rand_score(ensemble_labels_list[i], ensemble_labels_list[j])
        ari_matrix[i, j] = ari
        ari_matrix[j, i] = ari

# Show a summary: average ARI for each category of base clusterer
# Group by type
from collections import defaultdict
type_aris = defaultdict(list)
for i, desc in enumerate(ensemble_descriptions):
    for j, desc2 in enumerate(ensemble_descriptions):
        if i != j:
            if 'full' in desc and 'full' in desc2:
                type_aris['KMeans-full vs KMeans-full'].append(ari_matrix[i, j])
            elif 'Agglo' in desc and 'Agglo' in desc2:
                type_aris['Agglo vs Agglo'].append(ari_matrix[i, j])
            elif 'view' in desc and 'view' in desc2:
                type_aris['View vs View'].append(ari_matrix[i, j])

print(f'{"Pair type":<35s} {"Mean ARI":>10s} {"Interpretation":>30s}')
print('-' * 77)
for pair_type, aris in sorted(type_aris.items()):
    mean_ari = np.mean(aris)
    interp = 'High agreement' if mean_ari > 0.3 else 'Moderate diversity' if mean_ari > 0.1 else 'High diversity'
    print(f'{pair_type:<35s} {mean_ari:>10.4f} {interp:>30s}')

print(f'\nOverall mean pairwise ARI: {ari_matrix[np.triu_indices(n_base, k=1)].mean():.4f}')
print('(Lower = more diverse ensemble = typically better consensus)')

---
## Step 21 — Subspace Clustering

### The Curse of Dimensionality Problem

With 269 features, many clusters may only be meaningful in a **subset** of those features. For example:
- Horror movies might cluster tightly on "Category 07" and "Category 02" features, but be scattered across "Setting" features
- Romantic comedies might cluster on "Relationships" and "Dialogue" features, but not "Visual Effects"

Standard clustering uses ALL features equally, which dilutes these subspace-specific patterns with irrelevant noise.

### What is Subspace Clustering?

Subspace Clustering finds clusters that exist in **different subspaces** (subsets of features). Each cluster gets its own set of "relevant" features.

### Our approach: Soft Subspace Clustering (Feature Weighting per Cluster)

We implement a variant of **Weighted K-Means** where each cluster learns its own feature weights:
1. Start with standard K-Means clusters
2. For each cluster, compute how much each feature contributes to defining that cluster (low within-cluster variance → high weight)
3. Re-cluster using the learned weights
4. Repeat until convergence

This reveals which taxonomy categories are most important for each cluster — giving interpretable, category-aware groupings.

In [ ]:
# =============================================
# SUBSPACE CLUSTERING — Weighted K-Means (Entropy-Weighted)
# =============================================

def entropy_weighted_kmeans(X, n_clusters, n_iter=20, beta=3.0, random_state=42):
    """
    Entropy-Weighted K-Means (EW-KMeans).

    Each cluster maintains a weight vector over features.
    Features with low within-cluster dispersion get high weights.

    Parameters:
    -----------
    X : array (n_samples, n_features)
    n_clusters : int
    n_iter : number of iterations
    beta : controls weight sharpness (higher = sharper weights)

    Returns:
    --------
    labels : cluster assignments
    centers : cluster centroids
    weights : (n_clusters, n_features) weight matrix
    history : list of inertia values per iteration
    """
    np.random.seed(random_state)
    n, d = X.shape

    # Initialize with K-Means
    km_init = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=5)
    labels = km_init.fit_predict(X)
    centers = km_init.cluster_centers_.copy()

    # Initialize weights uniformly
    weights = np.ones((n_clusters, d)) / d

    history = []

    for iteration in range(n_iter):
        # --- Step 1: Update weights for each cluster ---
        for k in range(n_clusters):
            members = X[labels == k]
            if len(members) < 2:
                continue

            # Dispersion per feature (variance within cluster)
            dispersions = np.var(members, axis=0) + 1e-10  # avoid zero

            # Entropy-weighted: w_j = exp(-beta * D_j) / sum(exp(-beta * D_j))
            log_weights = -beta * dispersions
            log_weights -= log_weights.max()  # numerical stability
            w = np.exp(log_weights)
            w /= w.sum()
            weights[k] = w

        # --- Step 2: Assign each point to nearest cluster (weighted distance) ---
        # Weighted distance: d(x, c_k) = sum_j w_kj * (x_j - c_kj)^2
        new_labels = np.zeros(n, dtype=int)
        for i in range(n):
            dists = np.array([np.sum(weights[k] * (X[i] - centers[k])**2) for k in range(n_clusters)])
            new_labels[i] = np.argmin(dists)

        # --- Step 3: Update centers ---
        new_centers = np.zeros_like(centers)
        for k in range(n_clusters):
            members = X[new_labels == k]
            if len(members) > 0:
                new_centers[k] = members.mean(axis=0)
            else:
                new_centers[k] = centers[k]  # keep old center if empty

        # Compute weighted inertia
        inertia = 0
        for k in range(n_clusters):
            members = X[new_labels == k]
            if len(members) > 0:
                diffs = members - new_centers[k]
                inertia += np.sum(weights[k] * np.sum(diffs**2, axis=0))

        history.append(inertia)

        # Check convergence
        if np.array_equal(labels, new_labels):
            print(f'  Converged at iteration {iteration + 1}')
            break

        labels = new_labels
        centers = new_centers

    return labels, centers, weights, history

# Run Entropy-Weighted K-Means
SUBSPACE_K = N_FINAL_CLUSTERS
print(f'Running Entropy-Weighted K-Means (K={SUBSPACE_K}, beta=3.0)...')
sub_labels, sub_centers, sub_weights, sub_history = entropy_weighted_kmeans(
    X_scaled, n_clusters=SUBSPACE_K, n_iter=30, beta=3.0, random_state=42
)

sub_sil = silhouette_score(X_scaled, sub_labels, sample_size=5000, random_state=42)
print(f'\nSubspace Clustering silhouette: {sub_sil:.4f}')
print(f'\nCluster sizes:')
for cid in range(SUBSPACE_K):
    print(f'  Cluster {cid}: {(sub_labels == cid).sum():,} movies')

In [ ]:
# =============================================
# SUBSPACE CLUSTERING VISUALIZATIONS
# =============================================

# 1. Convergence plot
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

axes[0, 0].plot(range(1, len(sub_history)+1), sub_history, 'o-', color='steelblue', linewidth=2)
axes[0, 0].set_xlabel('Iteration', fontsize=11)
axes[0, 0].set_ylabel('Weighted Inertia', fontsize=11)
axes[0, 0].set_title('EW-KMeans Convergence', fontsize=13)

# 2. PCA projection
sub_palette = sns.color_palette('Set2', SUBSPACE_K)
for cid in range(SUBSPACE_K):
    mask = sub_labels == cid
    axes[0, 1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                        c=[sub_palette[cid]], alpha=0.4, s=12, edgecolors='none',
                        label=f'Cluster {cid} ({mask.sum():,})')
axes[0, 1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=11)
axes[0, 1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=11)
axes[0, 1].set_title('Subspace Clustering — PCA Projection', fontsize=13)
axes[0, 1].legend(fontsize=8, markerscale=3)

# 3. t-SNE projection
for cid in range(SUBSPACE_K):
    mask = sub_labels == cid
    axes[1, 0].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                        c=[sub_palette[cid]], alpha=0.4, s=12, edgecolors='none',
                        label=f'Cluster {cid} ({mask.sum():,})')
axes[1, 0].set_xlabel('t-SNE 1', fontsize=11)
axes[1, 0].set_ylabel('t-SNE 2', fontsize=11)
axes[1, 0].set_title('Subspace Clustering — t-SNE Projection', fontsize=13)
axes[1, 0].legend(fontsize=8, markerscale=3)

# 4. Feature weight heatmap by category
# Aggregate feature weights to category level for each cluster
all_cols = list(feature_data.columns)
cat_weight_matrix = np.zeros((SUBSPACE_K, len(category_features)))
cat_names_ordered = sorted(category_features.keys())

for k in range(SUBSPACE_K):
    for ci, cat in enumerate(cat_names_ordered):
        feats = category_features[cat]
        feat_indices = [all_cols.index(f) for f in feats if f in all_cols]
        if feat_indices:
            cat_weight_matrix[k, ci] = sub_weights[k, feat_indices].sum()

# Normalize each row to sum to 1 for comparison
row_sums = cat_weight_matrix.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
cat_weight_norm = cat_weight_matrix / row_sums

im = axes[1, 1].imshow(cat_weight_norm.T, cmap='YlOrRd', aspect='auto')
axes[1, 1].set_xticks(range(SUBSPACE_K))
axes[1, 1].set_xticklabels([f'C{k}' for k in range(SUBSPACE_K)], fontsize=10)
axes[1, 1].set_yticks(range(len(cat_names_ordered)))
axes[1, 1].set_yticklabels(cat_names_ordered, fontsize=8)
axes[1, 1].set_xlabel('Cluster', fontsize=11)
axes[1, 1].set_title('Category Importance per Cluster (Subspace Weights)', fontsize=13)
plt.colorbar(im, ax=axes[1, 1], label='Normalized weight')

plt.tight_layout()
plt.show()

In [ ]:
# Detailed subspace analysis: which categories define each cluster?
print('=' * 80)
print('SUBSPACE ANALYSIS: Which categories matter most for each cluster?')
print('=' * 80)

for k in range(SUBSPACE_K):
    n_members = (sub_labels == k).sum()
    print(f'\n--- Cluster {k} ({n_members:,} movies) ---')

    # Get category weights sorted
    cat_weights_k = [(cat_names_ordered[ci], cat_weight_norm[k, ci])
                     for ci in range(len(cat_names_ordered))]
    cat_weights_k.sort(key=lambda x: -x[1])

    print('  Top defining categories:')
    for cat, w in cat_weights_k[:5]:
        bar = '█' * int(w * 50)
        print(f'    {cat:<42s} {w:.3f} {bar}')

    # Also show top individual features
    top_feat_idx = np.argsort(sub_weights[k])[::-1][:8]
    print('  Top weighted individual features:')
    for idx in top_feat_idx:
        if idx < len(all_cols):
            fname = all_cols[idx]
            cat = feature_to_category.get(fname, 'Genre/Other')
            print(f'    {fname:<40s} [{cat}]  w={sub_weights[k, idx]:.4f}')

---
## Step 23 — Grand Comparison: Unified Elbow & Silhouette Analysis

Now we put every method on equal footing. For each clustering approach, we sweep K from 2 to 30 and compute **silhouette scores on the same original feature space** (the scaled+normalized 269-dimensional matrix). This ensures we're measuring the exact same thing regardless of what internal distance metric each method used.

The genre-only baseline is included as the floor to beat. Any method that consistently scores above the genre line is capturing structure beyond what genre labels alone provide.

### What the charts show

- **Silhouette vs K** — Higher is better. The line shows how well each method separates clusters at each K. Methods that stay above the genre baseline across many K values are adding real value.
- **Cross-method agreement** — ARI/NMI between every pair of methods. High agreement between different approaches means the structure is real, not an artifact of one algorithm's assumptions.

In [ ]:
# =============================================
# UNIFIED K-SWEEP: Silhouette in the same feature space for all methods
# =============================================
from sklearn.cluster import SpectralClustering

K_SWEEP = list(range(2, 31))
print(f'Running K-sweep from {K_SWEEP[0]} to {K_SWEEP[-1]} for all methods...')
print(f'All silhouettes computed on the SAME {X_scaled.shape[1]}-dim scaled feature space\n')

# Storage for results
sweep_results = {}

# --- 1. Genre-Only Baseline (cluster on genre features, evaluate on full features) ---
print('Genre-Only Baseline...')
genre_sweep_sil = []
for k in K_SWEEP:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = km.fit_predict(X_genre_scaled)
    # Evaluate on FULL feature space
    sil = silhouette_score(X_scaled, labels, sample_size=5000, random_state=42)
    genre_sweep_sil.append(sil)
sweep_results['Genre-Only Baseline'] = genre_sweep_sil
print(f'  Best: K={K_SWEEP[np.argmax(genre_sweep_sil)]}, sil={max(genre_sweep_sil):.4f}')

# --- 2. K-Means (full features) ---
print('K-Means (full features)...')
kmeans_sweep_sil = []
for k in K_SWEEP:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels, sample_size=5000, random_state=42)
    kmeans_sweep_sil.append(sil)
sweep_results['K-Means'] = kmeans_sweep_sil
print(f'  Best: K={K_SWEEP[np.argmax(kmeans_sweep_sil)]}, sil={max(kmeans_sweep_sil):.4f}')

# --- 3. Agglomerative (Ward, full features, full dataset) ---
print('Agglomerative (Ward)...')
agglo_sweep_sil = []
for k in K_SWEEP:
    agglo = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = agglo.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels, sample_size=5000, random_state=42)
    agglo_sweep_sil.append(sil)
sweep_results['Agglomerative (Ward)'] = agglo_sweep_sil
print(f'  Best: K={K_SWEEP[np.argmax(agglo_sweep_sil)]}, sil={max(agglo_sweep_sil):.4f}')

# --- 4. Hierarchical with Category-Specific Metrics ---
print('Hierarchical (Category Metrics)...')
hier_sweep_sil = []
for k in K_SWEEP:
    hier = AgglomerativeClustering(n_clusters=k, metric='precomputed', linkage='average')
    labels = hier.fit_predict(fused_distance)
    # Evaluate on FULL feature space (same as everyone else)
    sil = silhouette_score(X_scaled, labels, sample_size=5000, random_state=42)
    hier_sweep_sil.append(sil)
sweep_results['Hierarchical (Cat. Metrics)'] = hier_sweep_sil
print(f'  Best: K={K_SWEEP[np.argmax(hier_sweep_sil)]}, sil={max(hier_sweep_sil):.4f}')

# --- 5. Subspace (EW-KMeans) ---
print('Subspace (EW-KMeans)... (slower — fewer K values)')
sub_sweep_sil = []
sub_K_values = [k for k in K_SWEEP if k <= 20 or k % 5 == 0]  # skip some for speed
sub_sweep_k = []
for k in sub_K_values:
    labels, _, _, _ = entropy_weighted_kmeans(X_scaled, n_clusters=k, n_iter=15, beta=3.0, random_state=42)
    sil = silhouette_score(X_scaled, labels, sample_size=5000, random_state=42)
    sub_sweep_sil.append(sil)
    sub_sweep_k.append(k)
sweep_results['Subspace (EW-KMeans)'] = (sub_sweep_k, sub_sweep_sil)
best_sub_idx = np.argmax(sub_sweep_sil)
print(f'  Best: K={sub_sweep_k[best_sub_idx]}, sil={max(sub_sweep_sil):.4f}')

print('\nK-sweep complete!')

In [ ]:
# =============================================
# Multi-View and Ensemble are expensive to re-run at every K
# because they involve internal sub-clustering steps.
# We run them at a smaller set of K values.
# =============================================
print('Multi-View Clustering K-sweep (computationally intensive)...')
mv_sweep_k = [2, 4, 6, 8, 10, 12, 15, 20, 25, 30]
mv_sweep_sil = []

for k_final in mv_sweep_k:
    # Rebuild co-association from existing view labels (these don't change with K)
    # Then run spectral clustering at this K
    try:
        spec = SpectralClustering(n_clusters=k_final, affinity='precomputed',
                                   random_state=42, n_init=10)
        mv_labels_k = spec.fit_predict(co_assoc)
        sil = silhouette_score(X_scaled, mv_labels_k, sample_size=5000, random_state=42)
    except Exception:
        sil = float('nan')
    mv_sweep_sil.append(sil)

sweep_results['Multi-View'] = (mv_sweep_k, mv_sweep_sil)
best_mv_idx = np.argmax([s for s in mv_sweep_sil if not np.isnan(s)])
print(f'  Best: K={mv_sweep_k[best_mv_idx]}, sil={mv_sweep_sil[best_mv_idx]:.4f}')

print('Ensemble Clustering K-sweep...')
ens_sweep_k = [2, 4, 6, 8, 10, 12, 15, 20, 25, 30]
ens_sweep_sil = []

for k_final in ens_sweep_k:
    try:
        spec = SpectralClustering(n_clusters=k_final, affinity='precomputed',
                                   random_state=42, n_init=10)
        ens_labels_k = spec.fit_predict(ensemble_co_assoc)
        sil = silhouette_score(X_scaled, ens_labels_k, sample_size=5000, random_state=42)
    except Exception:
        sil = float('nan')
    ens_sweep_sil.append(sil)

sweep_results['Ensemble'] = (ens_sweep_k, ens_sweep_sil)
best_ens_idx = np.argmax([s for s in ens_sweep_sil if not np.isnan(s)])
print(f'  Best: K={ens_sweep_k[best_ens_idx]}, sil={ens_sweep_sil[best_ens_idx]:.4f}')

print('\nAll K-sweeps complete!')

In [ ]:
# =============================================
# UNIFIED SILHOUETTE COMPARISON PLOT
# =============================================
fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# Colors for each method
method_colors = {
    'Genre-Only Baseline': '#95a5a6',
    'K-Means': '#e74c3c',
    'Agglomerative (Ward)': '#3498db',
    'Hierarchical (Cat. Metrics)': '#9b59b6',
    'Multi-View': '#2ecc71',
    'Ensemble': '#e67e22',
    'Subspace (EW-KMeans)': '#1abc9c',
}

method_styles = {
    'Genre-Only Baseline': {'linestyle': '--', 'linewidth': 3, 'alpha': 0.8},
    'K-Means': {'linestyle': '-', 'linewidth': 1.5, 'alpha': 0.9},
    'Agglomerative (Ward)': {'linestyle': '-', 'linewidth': 1.5, 'alpha': 0.9},
    'Hierarchical (Cat. Metrics)': {'linestyle': '-', 'linewidth': 1.5, 'alpha': 0.9},
    'Multi-View': {'linestyle': '-', 'linewidth': 1.5, 'alpha': 0.9},
    'Ensemble': {'linestyle': '-', 'linewidth': 1.5, 'alpha': 0.9},
    'Subspace (EW-KMeans)': {'linestyle': '-', 'linewidth': 1.5, 'alpha': 0.9},
}

# --- Left panel: All methods silhouette vs K ---
ax = axes[0]
for method_name, data in sweep_results.items():
    color = method_colors.get(method_name, 'black')
    style = method_styles.get(method_name, {})

    if isinstance(data, tuple):
        # Sparse K values (Multi-View, Ensemble, Subspace)
        k_vals, sil_vals = data
        ax.plot(k_vals, sil_vals, 'o-', color=color, label=method_name,
                markersize=5, **style)
    else:
        # Full K range
        ax.plot(K_SWEEP, data, color=color, label=method_name, **style)

ax.set_xlabel('Number of Clusters (K)', fontsize=12)
ax.set_ylabel('Silhouette Score (full feature space)', fontsize=12)
ax.set_title('Silhouette Score vs K — All Methods', fontsize=14)
ax.legend(fontsize=9, loc='best')
ax.grid(True, alpha=0.3)

# --- Right panel: Improvement over genre baseline at each K ---
ax = axes[1]
# For methods with full K range
genre_sil_array = np.array(sweep_results['Genre-Only Baseline'])
for method_name, data in sweep_results.items():
    if method_name == 'Genre-Only Baseline':
        ax.axhline(y=0, color='#95a5a6', linestyle='--', linewidth=2, label='Genre Baseline (0)')
        continue

    color = method_colors.get(method_name, 'black')
    style = method_styles.get(method_name, {})

    if isinstance(data, tuple):
        k_vals, sil_vals = data
        # Interpolate genre baseline at these K values
        genre_at_k = np.interp(k_vals, K_SWEEP, genre_sil_array)
        improvement = np.array(sil_vals) - genre_at_k
        ax.plot(k_vals, improvement, 'o-', color=color, label=method_name,
                markersize=5, **style)
    else:
        improvement = np.array(data) - genre_sil_array
        ax.plot(K_SWEEP, improvement, color=color, label=method_name, **style)

ax.set_xlabel('Number of Clusters (K)', fontsize=12)
ax.set_ylabel('Silhouette Improvement over Genre Baseline', fontsize=12)
ax.set_title('How Much Does Each Method Beat Genre-Only?', fontsize=14)
ax.legend(fontsize=9, loc='best')
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='gray', linewidth=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================
# BEST-K SUMMARY TABLE
# =============================================
print('=' * 100)
print('BEST SILHOUETTE SCORE PER METHOD (evaluated on the same 269-dim feature space)')
print('=' * 100)

summary_rows = []
for method_name, data in sweep_results.items():
    if isinstance(data, tuple):
        k_vals, sil_vals = data
        best_idx = np.argmax(sil_vals)
        best_k = k_vals[best_idx]
        best_sil = sil_vals[best_idx]
    else:
        best_idx = np.argmax(data)
        best_k = K_SWEEP[best_idx]
        best_sil = data[best_idx]

    # Also get the silhouette at K=N_FINAL_CLUSTERS for direct comparison
    if isinstance(data, tuple):
        k_vals, sil_vals = data
        if N_FINAL_CLUSTERS in k_vals:
            matched_sil = sil_vals[k_vals.index(N_FINAL_CLUSTERS)]
        else:
            matched_sil = np.interp(N_FINAL_CLUSTERS, k_vals, sil_vals)
    else:
        matched_idx = K_SWEEP.index(N_FINAL_CLUSTERS) if N_FINAL_CLUSTERS in K_SWEEP else 0
        matched_sil = data[matched_idx]

    summary_rows.append({
        'Method': method_name,
        'Best K': best_k,
        'Best Silhouette': best_sil,
        f'Sil @ K={N_FINAL_CLUSTERS}': matched_sil,
    })

summary_df = pd.DataFrame(summary_rows).sort_values('Best Silhouette', ascending=False)

# Get genre baseline values for comparison
genre_best_sil = summary_df[summary_df['Method'] == 'Genre-Only Baseline']['Best Silhouette'].values[0]

print(f'\n{"Method":<30s} {"Best K":>7s} {"Best Sil":>10s} {"vs Genre":>10s} {f"Sil@K={N_FINAL_CLUSTERS}":>12s}')
print('-' * 72)
for _, row in summary_df.iterrows():
    improvement = row['Best Silhouette'] - genre_best_sil
    imp_str = f'+{improvement:.4f}' if improvement >= 0 else f'{improvement:.4f}'
    marker = ' ◄ BASELINE' if row['Method'] == 'Genre-Only Baseline' else ''
    print(f'{row["Method"]:<30s} {row["Best K"]:>7d} {row["Best Silhouette"]:>10.4f} {imp_str:>10s} {row[f"Sil @ K={N_FINAL_CLUSTERS}"]:>12.4f}{marker}')

# Bar chart
fig, ax = plt.subplots(figsize=(14, 5))
colors = [method_colors.get(row['Method'], '#333') for _, row in summary_df.iterrows()]
bars = ax.barh(range(len(summary_df)), summary_df['Best Silhouette'], color=colors, edgecolor='white')
ax.set_yticks(range(len(summary_df)))
ax.set_yticklabels([f"{row['Method']} (K={row['Best K']})" for _, row in summary_df.iterrows()], fontsize=10)
ax.set_xlabel('Best Silhouette Score', fontsize=12)
ax.set_title('Best Achievable Silhouette per Method', fontsize=14)
ax.invert_yaxis()

# Draw genre baseline line
ax.axvline(x=genre_best_sil, color='#95a5a6', linestyle='--', linewidth=2, alpha=0.8, label='Genre Baseline')
ax.legend(fontsize=10)

for bar, val in zip(bars, summary_df['Best Silhouette']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, f'{val:.4f}',
            va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# =============================================
# SIDE-BY-SIDE VISUAL COMPARISON (at matched K)
# =============================================
# Re-run each method at N_FINAL_CLUSTERS for direct visual comparison
print(f'Running all methods at K={N_FINAL_CLUSTERS} for side-by-side comparison...\n')

# Gather labels at the matched K
methods_matched = {}

# Genre baseline
methods_matched['Genre-Only\nBaseline'] = genre_match_labels

# K-Means
km_matched = KMeans(n_clusters=N_FINAL_CLUSTERS, random_state=42, n_init=20)
methods_matched['K-Means'] = km_matched.fit_predict(X_scaled)

# Agglomerative
agglo_matched = AgglomerativeClustering(n_clusters=N_FINAL_CLUSTERS, linkage='ward')
methods_matched['Agglomerative\n(Ward)'] = agglo_matched.fit_predict(X_scaled)

# Hierarchical (Category Metrics)
hier_matched = AgglomerativeClustering(n_clusters=N_FINAL_CLUSTERS, metric='precomputed', linkage='average')
methods_matched['Hierarchical\n(Cat. Metrics)'] = hier_matched.fit_predict(fused_distance)

# Multi-View (re-use co_assoc, just change K)
try:
    spec_mv = SpectralClustering(n_clusters=N_FINAL_CLUSTERS, affinity='precomputed', random_state=42, n_init=10)
    methods_matched['Multi-View'] = spec_mv.fit_predict(co_assoc)
except:
    methods_matched['Multi-View'] = multiview_labels

# Ensemble (re-use ensemble_co_assoc, just change K)
try:
    spec_ens = SpectralClustering(n_clusters=N_FINAL_CLUSTERS, affinity='precomputed', random_state=42, n_init=10)
    methods_matched['Ensemble'] = spec_ens.fit_predict(ensemble_co_assoc)
except:
    methods_matched['Ensemble'] = ensemble_labels

# Subspace
sub_labels_matched, _, _, _ = entropy_weighted_kmeans(X_scaled, n_clusters=N_FINAL_CLUSTERS, n_iter=20, beta=3.0, random_state=42)
methods_matched['Subspace\n(EW-KMeans)'] = sub_labels_matched

n_methods = len(methods_matched)
fig, axes = plt.subplots(2, n_methods, figsize=(4 * n_methods, 10))

for idx, (name, labels) in enumerate(methods_matched.items()):
    n_k = len(np.unique(labels))
    pal = sns.color_palette('Set2', n_k)

    # Compute silhouette on full feature space
    sil = silhouette_score(X_scaled, labels, sample_size=5000, random_state=42)

    # PCA (top row)
    ax = axes[0, idx]
    for cid in range(n_k):
        mask = labels == cid
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=[pal[cid % len(pal)]], alpha=0.3, s=8, edgecolors='none')
    ax.set_title(f'{name}\nsil={sil:.4f}', fontsize=11, fontweight='bold')
    if idx == 0:
        ax.set_ylabel('PCA', fontsize=12, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])

    # t-SNE (bottom row)
    ax = axes[1, idx]
    for cid in range(n_k):
        mask = labels == cid
        ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                   c=[pal[cid % len(pal)]], alpha=0.3, s=8, edgecolors='none')
    if idx == 0:
        ax.set_ylabel('t-SNE', fontsize=12, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle(f'All Methods at K={N_FINAL_CLUSTERS} — PCA & t-SNE', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================
# CROSS-METHOD AGREEMENT MATRIX (ARI & NMI) at matched K
# =============================================
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

method_names_m = list(methods_matched.keys())
method_names_clean_m = [n.replace('\n', ' ') for n in method_names_m]
n_m = len(method_names_m)

ari_cross = np.zeros((n_m, n_m))
nmi_cross = np.zeros((n_m, n_m))

for i in range(n_m):
    for j in range(n_m):
        ari_cross[i, j] = adjusted_rand_score(methods_matched[method_names_m[i]], methods_matched[method_names_m[j]])
        nmi_cross[i, j] = normalized_mutual_info_score(methods_matched[method_names_m[i]], methods_matched[method_names_m[j]])

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(ari_cross, annot=True, fmt='.3f', cmap='RdYlGn', ax=axes[0],
            xticklabels=method_names_clean_m, yticklabels=method_names_clean_m,
            vmin=-0.1, vmax=1.0, linewidths=0.5)
axes[0].set_title('Adjusted Rand Index (ARI)', fontsize=13)
axes[0].tick_params(axis='x', rotation=30)

sns.heatmap(nmi_cross, annot=True, fmt='.3f', cmap='RdYlGn', ax=axes[1],
            xticklabels=method_names_clean_m, yticklabels=method_names_clean_m,
            vmin=0, vmax=1.0, linewidths=0.5)
axes[1].set_title('Normalized Mutual Information (NMI)', fontsize=13)
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle(f'Cross-Method Agreement at K={N_FINAL_CLUSTERS}', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Interpretation:')
print('  ARI = 1.0 → perfect agreement  |  ARI ≈ 0 → random  |  ARI < 0 → worse than random')
print('  NMI = 1.0 → perfect agreement  |  NMI = 0 → no shared information')
print('\nKey question: How similar are the content-based methods to Genre-Only?')
print('Low agreement with Genre-Only = the method is finding DIFFERENT structure than genre labels.')

# Compute average agreement with genre baseline
genre_idx = method_names_clean_m.index('Genre-Only Baseline')
print(f'\nAverage ARI with Genre-Only Baseline:')
for i, name in enumerate(method_names_clean_m):
    if i != genre_idx:
        print(f'  {name:<35s} ARI={ari_cross[i, genre_idx]:.3f}  NMI={nmi_cross[i, genre_idx]:.3f}')

---
## Summary & Recommendations

### What the genre baseline tells us

The Genre-Only Baseline clusters movies using nothing but their genre labels (22 binary features like Action, Comedy, Drama). This is the conventional way movies are categorized. Every content-based method above is trying to find richer, more nuanced groupings using the 248 content features.

Look at the **silhouette comparison chart**: methods that consistently sit above the gray genre baseline line are proving that content features capture meaningful structure that genre labels miss. The **improvement plot** quantifies exactly how much better each method does.

### What each method revealed

- **K-Means** is the simple, fast workhorse — a good middle ground between speed and quality
- **Agglomerative (Ward)** uses a different merging strategy that can find different shapes of clusters
- **Hierarchical with Category Metrics** respects the nature of each feature type (Jaccard for binary settings, cosine for emotional tones) — look at the category contribution chart to see which feature types drive the most separation
- **Multi-View** gives each taxonomy category an equal vote — the view agreement analysis shows which categories are most informative
- **Ensemble** averages over many algorithms and parameters for the most robust result — high consensus strength means stable clusters
- **Subspace (EW-KMeans)** learns which features matter for each cluster individually — the category weight heatmap reveals *why* each cluster exists

### Which method should you use?

- For **interpretability**: Subspace — it tells you which features define each cluster
- For **robustness**: Ensemble — it averages out algorithmic biases
- For **proving content features add value**: Compare any method's silhouette against the genre baseline
- For **category-level insight**: Multi-View or Hierarchical with Category Metrics

The **cross-method agreement matrix** is the final key output. Methods that agree with each other but *disagree* with Genre-Only are finding genuinely new structure in the content features — which is the whole point of this project.